In [0]:
-- CreateLocationTypes — work-type classifier at LOCATION level (Casey/Jason design
-- 2026-07-14). New step between Locations_with_Sources and Locations_Mapped.
-- Rulings: n_refs + has_journal = merge_key group windows; new table (rebuild idiom);
-- LIVE (flipped 2026-07-14, Casey-confirmed): `type` carries the classifier verdict;
-- `classified_type` (same value) + `classified_rule` are the audit columns.
-- Preprint rule is GROUP-level: registrant-prefix DOI anywhere in the work_group marks
-- every location of that work preprint. Full daily recompute, no hash (Jason).
-- Classifier: frozen post-T5 CASE (163 rules; parity 52,383/52,383, synth 117/117,
-- escape PASS, gold 85.65 ratified at WORK grain — location-grain re-validation tracked).

In [0]:
CREATE OR REPLACE TABLE identifier('openalex' || :env_suffix || '.works.locations_w_types')
CLUSTER BY (best_doi, provenance, native_id)
TBLPROPERTIES (
  'delta.dataSkippingNumIndexedCols' = 40,
  'delta.deletedFileRetentionDuration' = '30 days',
  'delta.logRetentionDuration' = '30 days'
)
AS (

WITH loc AS (
  SELECT
    l.provenance, l.native_id, l.native_id_namespace, l.source_id, l.type AS existing_type,
    CASE WHEN coalesce(l.merge_key.doi,'') = '' AND coalesce(l.merge_key.pmid,'') = ''
              AND coalesce(l.merge_key.arxiv,'') = '' AND coalesce(l.merge_key.title_author,'') = ''
         THEN concat_ws('~', 'row', l.provenance, l.native_id_namespace, l.native_id)
         ELSE concat_ws('|', coalesce(l.merge_key.doi,''), coalesce(l.merge_key.pmid,''),
              coalesce(l.merge_key.arxiv,''), coalesce(l.merge_key.title_author,''))
    END AS work_group,  -- keyless rows get a per-row group: no '|||' mega-partition / feature leak
    l.title AS oa_title,
    l.raw_type AS oa_raw_type,
    CASE WHEN l.provenance = 'crossref' THEN l.raw_type END AS cr_type,
    CASE WHEN l.provenance = 'crossref' THEN l.source_name END AS cr_container,
    false AS cr_isbn,
    l.source_name AS oa_source_name,
    l.issue AS oa_issue, l.first_page AS oa_first_page,
    (l.first_page IS NOT NULL AND l.first_page <> '' AND l.first_page = l.last_page) AS oa_single_page,
    coalesce(size(l.references), 0) AS rec_n_refs,
    (l.abstract IS NOT NULL AND l.abstract <> '') AS oa_has_abstract,
    coalesce(l.is_retracted, false) AS oa_is_retracted,
    CAST(NULL AS STRING) AS oa_type,        -- recovery rule deliberately dormant (types being nulled)
    l.abstract AS oa_abstract,
    coalesce(l.best_doi, l.merge_key.doi, CASE WHEN l.native_id_namespace = 'doi' THEN l.native_id END) AS doi,
    l.landing_page_url AS tx_resolved_url
  FROM identifier('openalex' || :env_suffix || '.works.locations_w_sources') l
  
),
cr_sub AS (  -- subtype exists only in crossref_exploded (1 rule); crossref rows only
  SELECT native_id, max(subtype) AS cr_subtype
  FROM openalex.crossref.crossref_exploded
  WHERE native_id IN (SELECT native_id FROM loc WHERE provenance = 'crossref')
  GROUP BY native_id
),
src AS (
  SELECT id AS source_id, type AS src_type FROM openalex.sources.sources
),
works AS (
  SELECT
    concat_ws('~', loc.provenance, loc.native_id_namespace, loc.native_id) AS work_id,
    loc.provenance, loc.native_id, loc.native_id_namespace, loc.existing_type, loc.work_group,
    loc.oa_title, loc.oa_raw_type, loc.cr_type,
    CASE WHEN loc.provenance = 'crossref' THEN cs.cr_subtype END AS cr_subtype,
    loc.cr_container, loc.cr_isbn, loc.oa_source_name,
    s.src_type AS oa_source_type,
    loc.oa_issue, loc.oa_first_page,
    max(coalesce(loc.rec_n_refs, 0)) OVER (PARTITION BY loc.work_group) AS oa_n_refs,
    loc.oa_single_page, loc.oa_has_abstract, loc.oa_is_retracted, loc.oa_type, loc.oa_abstract,
    m.tx_meta AS tx_meta, m.tx_page_title AS tx_page_title,
    loc.tx_resolved_url, loc.doi,
    max(CASE WHEN lower(coalesce(s.src_type, '')) = 'journal' THEN 1 ELSE 0 END)
      OVER (PARTITION BY loc.work_group) = 1 AS oa_has_journal_location,
    max(CASE WHEN loc.doi LIKE '10.48550/%' OR loc.doi LIKE '10.1101/%'
          OR loc.doi LIKE '10.21203/rs.%' OR loc.doi LIKE '10.2139/ssrn.%'
          OR loc.doi LIKE '10.20944/preprints%' THEN 1 ELSE 0 END)
      OVER (PARTITION BY loc.work_group) = 1 AS preprint_registrant
  FROM loc
  LEFT JOIN cr_sub cs ON cs.native_id = loc.native_id AND loc.provenance = 'crossref'
  LEFT JOIN src s ON s.source_id = loc.source_id
  LEFT JOIN openalex.landing_page.meta_tags_for_work_type m
    ON m.native_id = CASE WHEN loc.native_id_namespace = 'doi'
                          THEN lower(loc.native_id) ELSE loc.native_id END
    AND m.native_id_namespace = loc.native_id_namespace  -- tags table stores DOIs lowercase; pmh handles case-preserved
),
feat AS (
  SELECT work_id,
    lower(coalesce(oa_title, '')) AS f_title,
    coalesce(oa_title, '') AS f_title_raw,  -- oxjob 940: original case for the title-code guard
    lower(coalesce(nullif(oa_raw_type, ''), nullif(cr_type, ''), '')) AS f_raw,
    lower(coalesce(cr_type, '')) AS f_crtype,
    lower(coalesce(cr_subtype, '')) AS f_sub,
    lower(coalesce(oa_source_name, '')) AS f_src,
    lower(coalesce(cr_container, '')) AS f_cont,
    lower(coalesce(cast(oa_issue AS string), '')) AS f_issue,
    CASE WHEN oa_first_page IS NULL THEN '' ELSE lower(trim(split(cast(oa_first_page AS string), '-')[0])) END AS f_fp,
    coalesce(oa_n_refs, 0) AS f_nrefs,
    coalesce(oa_single_page, false) AS f_single,
    coalesce(oa_has_abstract, false) AS f_hasabs,
    coalesce(oa_is_retracted, false) AS f_retr,
    lower(coalesce(oa_type, '')) AS f_oatype,
    lower(coalesce(tx_page_title, '')) AS f_ptl,
    lower(coalesce(oa_abstract, '')) AS f_abs,
    coalesce(regexp_extract(lower(coalesce(tx_resolved_url, '')), '^[a-z][a-z0-9+.\\-]*://([^/?#]*)', 1), '') AS f_host,
    CASE WHEN lower(coalesce(tx_resolved_url, '')) RLIKE '^[a-z][a-z0-9+.\\-]*://' THEN coalesce(regexp_extract(lower(coalesce(tx_resolved_url, '')), '^[a-z][a-z0-9+.\\-]*://[^/?#]*([^?#]*)', 1), '') ELSE lower(coalesce(tx_resolved_url, '')) END AS f_path,
    transform(flatten(transform(coalesce(tx_meta, array()), m -> regexp_extract_all(lower(m), '(?:dc\\.type(?:\\.articletype)?|article-type|articletype|dcterms\\.type|prism\\.contenttype|citation_article_type)"?\\s*(?:content=)?[":=]?\\s*"?\\s*([a-zA-Z][\\p{L}\\p{N}_ .\\-/]{1,40})', 1))), v -> trim(trim(TRAILING '"/' FROM trim(v)))) AS f_dc,
    transform(flatten(transform(coalesce(tx_meta, array()), m -> regexp_extract_all(lower(m), 'og:type"?\\s*(?:content=)?"?\\s*([a-zA-Z][\\p{L}\\p{N}_ .\\-/]{1,30})', 1))), v -> trim(trim(TRAILING '"/' FROM trim(v)))) AS f_og,
    exists(coalesce(tx_meta, array()), m -> lower(m) RLIKE '(?:name|property)\\s*=\\s*"(?:citation_conference_date|citation_conference_location)"') AS k_confabs,
    exists(coalesce(tx_meta, array()), m -> lower(m) RLIKE '(?:name|property)\\s*=\\s*"(?:citation_conference_abbrev|citation_conference_abbreviation|citation_conference_identifier|citation_conference_series_id)"') AS k_confpap,
    exists(coalesce(tx_meta, array()), m -> lower(m) RLIKE '(?:name|property)\\s*=\\s*"(?:bepress_citation_dissertation_institution|bepress_citation_dissertation_name|citation_dissertation_institution|citation_dissertation_name)"') AS k_diss,
    exists(coalesce(tx_meta, array()), m -> lower(m) LIKE '%citation_dissertation_%' AND lower(m) RLIKE 'content\\s*=\\s*"[^"]') AS k_diss_content,
    lower(coalesce(doi, '')) AS f_doi,
    lower(coalesce(tx_resolved_url, '')) AS f_url,
    coalesce(cr_isbn, false) AS f_isbn,
    lower(coalesce(oa_source_type, '')) AS f_srctype,
    coalesce(oa_has_journal_location, false) AS f_hasjournal,
    lower(array_join(coalesce(tx_meta, array()), ' ')) AS f_meta,
    exists(coalesce(tx_meta, array()), m -> lower(m) RLIKE '(?:name|property)\\s*=\\s*"citation_isbn"') AS k_isbn
  FROM works),
feat2 AS (
  SELECT *,
    concat(f_src, ' ', f_cont) AS f_sc,
    regexp_extract_all(f_path, '[a-z]{3,}', 0) AS f_urltok,
    replace(replace(replace(replace(f_raw, '-', ''), '_', ''), ' ', ''), ':', '') AS f_rawnorm
  FROM feat),
scored AS (
  SELECT work_id,
  CASE
      WHEN (f_title LIKE 'retraction%' OR f_title LIKE 'statement of retraction%') OR (f_retr AND f_title LIKE 'withdrawn%') OR (f_abs LIKE '%this retracts%' OR f_abs LIKE '%retracts the article%') THEN 'retraction'
      WHEN (f_title LIKE '%erratum%' OR f_title LIKE '%corrigendum%' OR f_title LIKE '%correction to%' OR f_title LIKE '%author correction%' OR f_title LIKE '%publisher correction%') OR f_title LIKE 'correction%' OR (f_abs LIKE '%this corrects the article%' OR f_abs LIKE '%corrects the article%') THEN 'erratum'
      WHEN f_rawnorm = 'peerreview' OR (f_title LIKE 'review for%' OR f_title LIKE 'decision letter%' OR f_title LIKE 'author response%' OR f_title LIKE 'reply on%' OR f_title LIKE 'peer review of%' OR f_title LIKE 'reviewer public%' OR f_title LIKE 'comment on egusphere%') THEN 'peer-review'
      WHEN f_crtype = 'dissertation' THEN 'dissertation'
      WHEN f_crtype IN ('reference-entry','reference-book') THEN 'reference-entry'
      WHEN f_crtype = 'standard' THEN 'standard'
      WHEN f_crtype = 'report-component' THEN 'report'
      WHEN f_sub = 'preprint' THEN 'preprint'
      WHEN f_host IN ('osf.io', 'www.researchsquare.com') THEN 'preprint'
      WHEN f_host IN ('www.encodeproject.org', 'www.rcsb.org', 'www.wwpdb.org') THEN 'dataset'
      WHEN f_host IN ('www.softxjournal.com') THEN 'software-paper'
      WHEN (f_host IN ('cran.r-project.org', 'demonstrations.wolfram.com')) AND f_raw <> 'dataset' THEN 'software'
      WHEN f_host IN ('facultyopinions.com', 'publons.com', 'www.webofscience.com') THEN 'peer-review'
      WHEN f_host IN ('theses.fr', 'theses.hal.science') THEN 'dissertation'
      WHEN f_host IN ('materials.springer.com', 'referenceworks.brill.com', 'www.cabidigitallibrary.org', 'www.oed.com', 'www.oxfordartonline.com', 'www.ukwhoswho.com') THEN 'reference-entry'
      WHEN f_host IN ('meetingorganizer.copernicus.org', 'www.morressier.com') THEN 'conference-abstract'
      WHEN f_host IN ('goodreads.com', 'www.goodreads.com') THEN 'book'
      WHEN f_host IN ('picryl.com', 'www.picryl.com') THEN 'other'
      WHEN f_src IN ('abstracts', 'abstracts with programs - geological society of america', 'academy of management proceedings', 'endocrine abstracts', 'the proceedings of the annual convention of the japanese psychological association') THEN 'conference-abstract'
      WHEN f_src IN ('brill’s new pauly', 'definitions', 'der neue pauly', 'encyclopédie de l’islam', 'iucn red list of threatened species', 'lexikon des gesamten buchwesens online', 'radiopaedia.org', 'religion in geschichte und gegenwart', 'springerreference', 'supplementum epigraphicum graecum', 'the shafr guide online', 'who was who', 'who\'s who') THEN 'reference-entry'
      WHEN f_src IN ('psyctests dataset') THEN 'dataset'
      WHEN f_src IN ('research square', 'ssrn electronic journal') THEN 'preprint'
      WHEN f_src IN ('data in brief') THEN 'data-paper'
      WHEN f_src IN ('softwarex', 'the journal of open source software') THEN 'software-paper'
      WHEN f_src IN ('acta horticulturae', 'ecs transactions', 'iceri proceedings', 'ifac proceedings volumes', 'materials today proceedings', 'procedia engineering') THEN 'conference-paper'
      WHEN f_src IN ('faculty opinions – post-publication peer review of the biomedical literature') THEN 'peer-review'
      WHEN f_src IN ('apress ebooks', 'jaypee brothers medical publishers (p) ltd. ebooks') THEN 'book-chapter'
      WHEN f_src IN ('bulletin of the center for children\'s books', 'choice reviews online') THEN 'book-review'
      WHEN f_src IN ('electronic enlightenment scholarly edition of correspondence') THEN 'other'
      WHEN f_src IN ('national bureau of economic research') THEN 'report'
      WHEN f_src IN ('synfacts') THEN 'editorial'
      WHEN f_sc LIKE '%datasets%' THEN 'dataset'
      WHEN f_sc LIKE '%web of conferences%' THEN 'conference-paper'
      WHEN f_sc LIKE '%rxiv%' THEN 'preprint'
      WHEN f_sc LIKE '%preprint%' THEN 'preprint'
      WHEN f_sc LIKE '%dictionary%' THEN 'reference-entry'
      WHEN f_sc LIKE '%encyclopedia%' THEN 'reference-entry'
      WHEN f_sc LIKE '%lexicon%' THEN 'reference-entry'
      WHEN f_sc LIKE '%meeting abstracts%' THEN 'conference-abstract'
      WHEN f_src IN ('e3s web of conferences', 'lecture notes on data engineering and communications technologies', 'procedia - social and behavioral sciences') THEN 'conference-paper'
      WHEN f_src IN ('european urology supplements') THEN 'conference-abstract'
      WHEN f_src IN ('gisaid') THEN 'dataset'
      WHEN (f_src LIKE '%encode%' OR f_cont LIKE '%encode%') THEN 'dataset'
      WHEN (f_src LIKE '%spie proceedings%' OR f_cont LIKE '%spie proceedings%') THEN 'conference-paper'
      WHEN (f_src LIKE '%worldwide protein data bank%' OR f_cont LIKE '%worldwide protein data bank%') THEN 'dataset'
      WHEN (f_src LIKE '%sae technical paper series%' OR f_cont LIKE '%sae technical paper series%') THEN 'conference-paper'
      WHEN (f_src LIKE '%advances in social science, education and humanities research%' OR f_cont LIKE '%advances in social science, education and humanities research%') THEN 'conference-paper'
      WHEN (f_src LIKE '%conference on lasers and electro-optics%' OR f_cont LIKE '%conference on lasers and electro-optics%') THEN 'conference-paper'
      WHEN (f_src LIKE '%ifmbe proceedings%' OR f_cont LIKE '%ifmbe proceedings%') THEN 'conference-paper'
      WHEN (f_src LIKE '%morphosource%' OR f_cont LIKE '%morphosource%') THEN 'dataset'
      WHEN (f_src LIKE '%sgem international multidisciplinary scientific geoconference%' OR f_cont LIKE '%sgem international multidisciplinary scientific geoconference%') THEN 'conference-paper'
      WHEN f_doi LIKE '%meetingabstracts%' OR f_doi LIKE '%meeting-abstracts%' OR f_url LIKE '%meetingabstracts%' OR f_url LIKE '%meeting-abstracts%' THEN 'conference-abstract'
      WHEN f_title LIKE 'editorial board%' THEN 'paratext'
      WHEN f_title LIKE 'front matter%' THEN 'paratext'
      WHEN (f_title LIKE 'preface%' OR f_title LIKE 'appendix%' OR f_title LIKE 'proofs of%') AND (f_raw IN ('book-chapter','book-part','chapter','book-section') OR f_crtype IN ('book-chapter','monograph','edited-book')) THEN 'paratext'
      WHEN array_contains(f_urltok, 'referenceworkentry') THEN 'reference-entry'
      WHEN array_contains(f_urltok, 'meetingabstracts') THEN 'conference-abstract'
      WHEN (array_contains(f_urltok, 'thesis') OR array_contains(f_urltok, 'theses') OR array_contains(f_urltok, 'dissertations')) AND f_crtype = '' AND f_srctype <> 'journal' THEN 'dissertation'
      WHEN k_confabs THEN 'conference-abstract'
      WHEN k_confpap THEN 'conference-paper'
      WHEN array_contains(f_dc, 'book-review') THEN 'book-review'
      WHEN array_contains(f_dc, 'bookreview') THEN 'book-review'
      WHEN array_contains(f_dc, 'book reviews') THEN 'book-review'
      WHEN array_contains(f_dc, 'book review') THEN 'book-review'
      WHEN array_contains(f_dc, 'reseñas') THEN 'book-review'
      WHEN array_contains(f_dc, 'thesis') THEN 'dissertation'
      WHEN array_contains(f_dc, 'dissertação') THEN 'dissertation'
      WHEN array_contains(f_dc, 'doctoral dissertation') THEN 'dissertation'
      WHEN array_contains(f_dc, 'pg_thesis') THEN 'dissertation'
      WHEN array_contains(f_dc, 'editorial') THEN 'editorial'
      WHEN array_contains(f_dc, 'editorialnotes') THEN 'editorial'
      WHEN array_contains(f_dc, 'article-commentary') THEN 'editorial'
      WHEN array_contains(f_dc, 'meeting-report') THEN 'conference-abstract'
      WHEN array_contains(f_dc, 'congress-abstract') THEN 'conference-abstract'
      WHEN array_contains(f_dc, 'oxan-executive-summary') THEN 'report'
      WHEN array_contains(f_dc, 'news') THEN 'other'
      WHEN array_contains(f_dc, 'chapter') THEN 'book-chapter'
      WHEN f_ptl LIKE 'reply%' THEN 'letter'
      WHEN (f_title LIKE 'supplementary%' OR f_title LIKE 'supplemental%' OR f_title LIKE 'figure from%') OR (f_title LIKE '%supplementary figure%' OR f_title LIKE '%supplementary table%' OR f_title LIKE '%supplemental material%' OR f_title LIKE '%figure from%') THEN 'supplementary-materials'
      WHEN (f_title LIKE 'table of contents%' OR f_title LIKE 'contents%' OR f_title LIKE 'front matter%' OR f_title LIKE 'back matter%' OR f_title LIKE 'frontmatter%' OR f_title LIKE 'front cover%' OR f_title LIKE 'editorial board%' OR f_title LIKE 'subject index%' OR f_title LIKE 'author index%' OR f_title LIKE 'name index%' OR f_title LIKE 'list of figures%' OR f_title LIKE 'list of tables%' OR f_title LIKE 'list of contributors%' OR f_title LIKE 'list of abbreviations%' OR f_title LIKE 'list of illustrations%' OR f_title LIKE 'list of plates%' OR f_title LIKE 'bibliography%' OR f_title LIKE 'abbreviations%' OR f_title LIKE 'abbreviation%' OR f_title LIKE 'acknowledgment%' OR f_title LIKE 'acknowledgments%' OR f_title LIKE 'acknowledgement%' OR f_title LIKE 'acknowledgements%' OR f_title LIKE 'dedication%' OR f_title LIKE 'contributors%' OR f_title LIKE 'about the author%' OR f_title LIKE 'about the editor%' OR f_title LIKE 'copyright%' OR f_title LIKE 'title page%' OR f_title LIKE 'masthead%' OR f_title LIKE 'frontispiece%' OR f_title LIKE 'titelei%' OR f_title LIKE 'inhaltsverzeichnis%' OR f_title LIKE 'sachregister%' OR f_title LIKE 'literaturverzeichnis%' OR f_title LIKE 'inhalt%' OR f_title LIKE 'session details%' OR f_title LIKE 'forthcoming%' OR f_title LIKE 'calendar%' OR f_title LIKE 'general index%' OR f_title LIKE 'back cover%' OR f_title LIKE 'inside front cover%' OR f_title LIKE 'prelims%' OR f_title LIKE 'preliminary material%' OR f_title LIKE 'backmatter%' OR f_title LIKE 'books received%' OR f_title LIKE 'works cited%' OR f_title LIKE 'about the contributors%' OR f_title LIKE 'author biograph%' OR f_title LIKE 'expediente%' OR f_title LIKE 'table des mati%' OR f_title LIKE 'remerciements%') THEN 'paratext'
      WHEN (f_title LIKE '%issue information%' OR f_title LIKE '%masthead%' OR f_title LIKE '%editorial board%' OR f_title LIKE '%instructions for authors%' OR f_title LIKE '%list of reviewers%' OR f_title LIKE '%acknowledgment of reviewers%' OR f_title LIKE '%acknowledgement of reviewers%' OR f_title LIKE '%cover image%' OR f_title LIKE '%information for authors%' OR f_title LIKE '%society information%' OR f_title LIKE '%information for contributors%' OR f_title LIKE '%information for readers%' OR f_title LIKE '%notes for contributors%' OR f_title LIKE '%notes on contributors%' OR f_title LIKE '%call for papers%' OR f_title LIKE '%call for submissions%' OR f_title LIKE '%call for abstracts%' OR f_title LIKE '%guide for authors%' OR f_title LIKE '%impressum%' OR f_title LIKE '%publication information%' OR f_title LIKE '%reviewer acknowledgement%' OR f_title LIKE '%additional journal content%' OR f_title LIKE '%[advertisement]%' OR f_title LIKE '%[advertisements]%') THEN 'paratext'
      WHEN trim(f_title) RLIKE '^[a-z]+ (19|20)[0-9]{2} placement ads$' THEN 'paratext'
      WHEN trim(f_title) = 'notes' THEN 'paratext'
      WHEN trim(f_title) = 'peer review statement' THEN 'paratext'
      WHEN (f_title LIKE 'program committee%' OR f_title LIKE 'organizing committee%' OR f_title LIKE 'workshop committee%' OR f_title LIKE 'conference committee%' OR f_title LIKE 'scientific committee%' OR f_title LIKE 'technical program committee%' OR f_title LIKE 'steering committee%') OR trim(f_title) RLIKE '^(program |organizing |scientific |technical |workshop |conference |steering )?committee(s)?( members| list(ing)?s?)?$' THEN 'paratext'
      WHEN f_title LIKE 'index%' OR ((f_title LIKE 'references%' OR f_title LIKE 'list of%') AND (f_fp IN ('i','ii','iii','iv','ix','v','vi','vii','viii','x','xi','xii','xiii','xiv','xv') OR f_nrefs = 0 OR NOT f_hasabs)) THEN 'paratext'
      WHEN (f_title LIKE '%python package%') THEN 'software-paper'
      WHEN (f_title LIKE 'din en%' OR f_title LIKE 'specification for%' OR f_title LIKE 'test method%') OR (f_title LIKE '%englische fassung%') THEN 'standard'
      WHEN (f_title LIKE 'encsr%') THEN 'dataset'
      WHEN (f_title LIKE 'book review%' OR f_title LIKE 'review of the book%' OR f_title LIKE 'reseña del libro%') OR (f_title LIKE '% isbn%' OR f_title LIKE '%edited by%' OR f_title LIKE '%[book review%' OR f_title LIKE '%(book review%') OR array_contains(f_dc, 'book-review') OR (f_title LIKE '%pp.%' AND (f_title LIKE '%isbn%' OR f_title LIKE '%press%' OR f_title LIKE '%£%')) THEN 'book-review'
      WHEN (f_title LIKE 'guest editorial%' OR f_title LIKE 'editorial comment%' OR f_title LIKE 'guest editor%' OR f_title LIKE 'commentary on%' OR f_title LIKE 'message from%' OR f_title LIKE 'editorial board is%' OR f_title LIKE 'editorial:%' OR f_title LIKE 'preface:%' OR f_title LIKE 'préambule%' OR f_title LIKE 'éditorial%' OR f_title LIKE 'editors\' note%' OR f_title LIKE 'editors note%' OR f_title LIKE 'special thanks%' OR f_title LIKE 'nota de la directora%' OR f_title LIKE 'note from the editor%' OR f_title LIKE 'interview with%' OR f_title LIKE 'interview:%' OR f_title LIKE 'entrevista%') OR (f_title LIKE '%from the editor%' OR f_title LIKE '%special issue on%' OR f_title LIKE '%to the special issue%' OR f_title LIKE '%commentary:%') OR (f_title LIKE 'editorial%' AND f_title NOT LIKE '%board%') THEN 'editorial'
      WHEN trim(f_title) IN ('presentación', 'presentación.', 'presentacion', 'presentacion.', 'presentació', 'presentació.') AND f_raw NOT LIKE '%book%' AND f_raw NOT LIKE '%chapter%' AND f_raw NOT LIKE '%capítulo%' AND f_raw NOT LIKE '%libro%' AND f_raw NOT LIKE '%conference%' AND f_raw NOT LIKE '%thesis%' AND f_raw NOT LIKE '%presentation%' AND f_raw NOT LIKE '%ponencia%' AND f_raw NOT LIKE 'tesis%' AND f_raw NOT IN ('dataset','database') AND f_crtype NOT IN ('book-chapter','monograph','edited-book') THEN 'editorial'
      WHEN (f_title LIKE 'letter to the%' OR f_title LIKE 'reply to%' OR f_title LIKE 'in reply%' OR f_title LIKE 'reader response%' OR f_title LIKE 'comments on the article%') OR (f_title LIKE '%to the editor%' OR f_title LIKE '%authors\' reply%' OR f_title LIKE '%reply to comment%') OR ((f_title LIKE 'reply%' OR f_title LIKE 'comment on%') AND f_single) OR f_title LIKE 'correspondence%' THEN 'letter'
      WHEN (f_title LIKE '%narrative review%' OR f_title LIKE '%mini-review%' OR f_title LIKE '%meta-analysis of%') THEN 'review'
      WHEN (f_title LIKE 'libguides%' OR f_title LIKE 'all guides%' OR f_title LIKE 'research guides%') THEN 'libguides'
      WHEN (f_title LIKE 're:%' OR f_title LIKE 'the authors reply%' OR f_title LIKE 'comment on:%') THEN 'letter'
      WHEN f_title LIKE 'discussion of%' THEN 'editorial'
      WHEN f_title LIKE 'data for %' THEN 'dataset'
      WHEN f_title LIKE '%systematic literature review%' AND NOT (f_title LIKE '%case report%' OR f_title LIKE '%case study%') THEN 'review'
      WHEN (f_title LIKE '%in memoriam%' OR f_title LIKE '%autograph letter%' OR f_title LIKE '%obituary%') THEN 'other'
      WHEN f_title LIKE 'abstract%' THEN 'conference-abstract'
      WHEN (f_src LIKE '%abstract%' OR f_cont LIKE '%abstract%') AND (f_single OR (f_nrefs = 0 AND f_hasabs)) THEN 'conference-abstract'
      WHEN f_src LIKE '%supplement%' AND f_single AND f_nrefs = 0 THEN 'conference-abstract'
      WHEN f_issue LIKE '%suppl%' AND f_single THEN 'conference-abstract'
      WHEN f_raw = 'journal-article' AND f_nrefs = 0 AND f_single AND (f_issue RLIKE '^s[0-9]' OR f_issue RLIKE '^[0-9]+s$') THEN 'conference-abstract'
      WHEN (f_abs LIKE '%abstracts of presentations%' OR f_abs LIKE '%searchable abstracts%') THEN 'conference-abstract'
      WHEN ltrim(f_abs) LIKE 'reviewed by%' THEN 'book-review'
      WHEN (f_abs LIKE '%this data article%') THEN 'data-paper'
      WHEN (f_abs LIKE '%this editorial%' OR f_abs LIKE '%in this editorial%') THEN 'editorial'
      WHEN f_src IN ('communications in computer and information science', 'energy procedia', 'lecture notes in civil engineering', 'lecture notes in computer science', 'procedia computer science') AND (f_nrefs = 0 AND f_single AND f_hasabs) THEN 'conference-abstract'
      WHEN f_src IN ('communications in computer and information science', 'energy procedia', 'lecture notes in civil engineering', 'lecture notes in computer science', 'procedia computer science') THEN 'conference-paper'
      WHEN f_src IN ('scientific data') THEN 'data-paper'
      WHEN (f_src LIKE '%journal of physics: conference series%' OR f_cont LIKE '%journal of physics: conference series%') AND (f_nrefs = 0 AND f_single AND f_hasabs) THEN 'conference-abstract'
      WHEN (f_src LIKE '%journal of physics: conference series%' OR f_cont LIKE '%journal of physics: conference series%') THEN 'conference-paper'
      WHEN f_title RLIKE '^[a-z]{1,3}-?[0-9]{2,5}[.:\\s\\p{Z}]' AND f_nrefs = 0 AND f_raw NOT IN ('dataset','database') AND ((f_title_raw RLIKE '^[A-Z]{1,3}-?[0-9]{2,5}[.:\\s\\p{Z}]' AND NOT f_title_raw RLIKE '^(CD|IL|HSP|FGF|SOX|MMP|MIR|PSD|BCL|TNF|HLA|EGF|TGF)-?[0-9]') OR f_title_raw RLIKE '^(Mo|Tu|We|Th|Fr|Sa|Su|Mon|Tue|Wed|Thu|Fri|Sat|Sun)-?[0-9]{3,5}[.:\\s\\p{Z}]' OR f_title_raw RLIKE '^[A-Za-z]{1,3}-?[0-9]{4,5}[.:\\s\\p{Z}]') THEN 'conference-abstract'
      WHEN f_title LIKE '%systematic review%' AND f_nrefs > 0 THEN 'review'
      WHEN f_oatype = 'review' AND f_nrefs >= 25 AND f_hasabs THEN 'review'
      WHEN f_sc LIKE '%conference%' AND (f_nrefs = 0 AND f_single AND f_hasabs) THEN 'conference-abstract'
      WHEN f_sc LIKE '%conference%' THEN 'conference-paper'
      WHEN f_sc LIKE '%symposium%' AND (f_nrefs = 0 AND f_single AND f_hasabs) THEN 'conference-abstract'
      WHEN f_sc LIKE '%symposium%' THEN 'conference-paper'
      WHEN f_sc LIKE '%workshop%' AND (f_nrefs = 0 AND f_single AND f_hasabs) THEN 'conference-abstract'
      WHEN f_sc LIKE '%workshop%' THEN 'conference-paper'
      WHEN f_raw = 'proceedings-article' AND (f_nrefs = 0 AND f_single AND f_hasabs) THEN 'conference-abstract'
      WHEN f_raw = 'proceedings-article' THEN 'conference-paper'
      WHEN f_raw = 'proceedings' AND f_crtype = '' AND f_title NOT LIKE 'proceedings%' AND (f_nrefs = 0 AND f_single AND f_hasabs) THEN 'conference-abstract'
      WHEN f_raw = 'proceedings' AND f_crtype = '' AND f_title NOT LIKE 'proceedings%' THEN 'conference-paper'
      WHEN f_doi LIKE '10.1596/1813-9450-%' THEN 'report'
      WHEN f_crtype = 'journal-issue' THEN 'paratext'
      WHEN f_crtype IN ('edited-book','monograph') THEN 'book'
      WHEN f_raw = 'reference-entry' THEN 'reference-entry'
      WHEN f_raw = 'dissertation' THEN 'dissertation'
      WHEN f_nrefs >= 20 AND (rtrim(f_title, ' .') LIKE '%a review' OR rtrim(f_title, ' .') LIKE '%a literature review' OR f_title LIKE '%scientometric review%') THEN 'review'
      WHEN f_title LIKE '%a meta-analysis%' AND f_nrefs >= 20 THEN 'review'
      WHEN f_raw LIKE '%eu-repo/semantics/%' AND trim(f_raw) LIKE '%/conferenceobject' THEN 'conference-paper'
      WHEN f_raw LIKE '%eu-repo/semantics/%' AND trim(f_raw) LIKE '%/bookpart' THEN 'book-chapter'
      WHEN f_raw LIKE '%eu-repo/semantics/%' AND trim(f_raw) LIKE '%/doctoralthesis' THEN 'dissertation'
      WHEN f_raw LIKE '%eu-repo/semantics/%' AND trim(f_raw) LIKE '%/masterthesis' THEN 'dissertation'
      WHEN f_raw LIKE '%eu-repo/semantics/%' AND trim(f_raw) LIKE '%/article' THEN 'article'
      WHEN f_raw LIKE '%eu-repo/semantics/%' AND trim(f_raw) LIKE '%/report' THEN 'report'
      WHEN f_raw LIKE '%eu-repo/semantics/%' AND trim(f_raw) LIKE '%/other' THEN 'other'
      WHEN f_raw LIKE '%thesis%' THEN 'dissertation'
      WHEN f_raw LIKE '%väitöskirja%' THEN 'dissertation'
      WHEN f_raw LIKE '%hochschulschrift%' THEN 'dissertation'
      WHEN (f_raw LIKE 'tesis%' OR f_raw LIKE '%bakalářská práce%') THEN 'dissertation'
      WHEN f_raw LIKE '%final year project%' THEN 'report'
      WHEN f_rawnorm IN ('chapter','bookpart') THEN 'book-chapter'
      WHEN f_rawnorm LIKE '%conferencepaper' THEN 'conference-paper'
      WHEN f_rawnorm = 'researchreport' THEN 'report'
      WHEN f_raw = 'figure' THEN 'supplementary-materials'
      WHEN f_rawnorm = 'software,multimedia' THEN 'other'
      WHEN f_raw = 'software' THEN 'software'
      WHEN f_raw LIKE '%printed serial%' THEN 'other'
      WHEN f_rawnorm IN ('image','physicalobject') THEN 'other'
      WHEN f_rawnorm IN ('audiovisual','sound') THEN 'other'
      WHEN (f_raw LIKE '%monograf%' OR f_raw LIKE '%monograph%') THEN 'book'
      WHEN f_rawnorm LIKE '%book' AND f_raw NOT IN ('book','edited-book','monograph','book-set') THEN 'book'
      WHEN f_raw LIKE '%preprint%' AND NOT (f_raw LIKE '%eu-repo%' AND NOT trim(f_raw) LIKE '%/preprint') AND NOT (f_srctype = 'journal' AND NOT (f_src LIKE '%rxiv%' OR f_src LIKE '%preprint%' OR f_src LIKE '%repec%' OR f_src LIKE '%ssrn%' OR f_src LIKE '%zenodo%' OR f_src LIKE '%research square%' OR f_src LIKE '%osf%')) AND NOT f_hasjournal THEN 'preprint'
      WHEN (trim(f_title) RLIKE '[0-9] ?(pp|pág(s|inas)?)\\.?$' OR f_title LIKE '%(isbn%') AND f_raw NOT IN ('book-chapter','book-part','book-section','book','edited-book','monograph','book-set','report','dataset','database','proceedings','posted-content') THEN 'book-review'
      WHEN f_raw IN ('book-chapter','book-part') THEN 'book-chapter'
      WHEN f_raw = 'book-section' THEN 'reference-entry'
      WHEN f_raw IN ('book','edited-book','monograph','book-set') THEN 'book'
      WHEN f_raw = 'report' THEN 'report'
      WHEN f_raw = 'posted-content' THEN 'other'
      WHEN f_raw IN ('dataset','database') THEN 'dataset'
      WHEN f_raw = 'proceedings' THEN 'paratext'
      WHEN f_raw = 'other' THEN 'other'
      ELSE 'article' END AS cascade_type,
  CASE
      WHEN (f_title LIKE 'retraction%' OR f_title LIKE 'statement of retraction%') OR (f_retr AND f_title LIKE 'withdrawn%') OR (f_abs LIKE '%this retracts%' OR f_abs LIKE '%retracts the article%') THEN 'retraction: dc.type / title-start'
      WHEN (f_title LIKE '%erratum%' OR f_title LIKE '%corrigendum%' OR f_title LIKE '%correction to%' OR f_title LIKE '%author correction%' OR f_title LIKE '%publisher correction%') OR f_title LIKE 'correction%' OR (f_abs LIKE '%this corrects the article%' OR f_abs LIKE '%corrects the article%') THEN 'erratum: title / dc.type'
      WHEN f_rawnorm = 'peerreview' OR (f_title LIKE 'review for%' OR f_title LIKE 'decision letter%' OR f_title LIKE 'author response%' OR f_title LIKE 'reply on%' OR f_title LIKE 'peer review of%' OR f_title LIKE 'reviewer public%' OR f_title LIKE 'comment on egusphere%') THEN 'peer-review: raw/title/dc'
      WHEN f_crtype = 'dissertation' THEN 'cr=dissertation'
      WHEN f_crtype IN ('reference-entry','reference-book') THEN 'cr=reference-entry'
      WHEN f_crtype = 'standard' THEN 'cr=standard'
      WHEN f_crtype = 'report-component' THEN 'cr=report-component'
      WHEN f_sub = 'preprint' THEN 'cr_subtype=preprint'
      WHEN f_host IN ('osf.io', 'www.researchsquare.com') THEN 'URL host -> type'
      WHEN f_host IN ('www.encodeproject.org', 'www.rcsb.org', 'www.wwpdb.org') THEN 'URL host -> type'
      WHEN f_host IN ('www.softxjournal.com') THEN 'URL host -> type'
      WHEN (f_host IN ('cran.r-project.org', 'demonstrations.wolfram.com')) AND f_raw <> 'dataset' THEN 'URL host -> type'
      WHEN f_host IN ('facultyopinions.com', 'publons.com', 'www.webofscience.com') THEN 'URL host -> type'
      WHEN f_host IN ('theses.fr', 'theses.hal.science') THEN 'URL host -> type'
      WHEN f_host IN ('materials.springer.com', 'referenceworks.brill.com', 'www.cabidigitallibrary.org', 'www.oed.com', 'www.oxfordartonline.com', 'www.ukwhoswho.com') THEN 'URL host -> type'
      WHEN f_host IN ('meetingorganizer.copernicus.org', 'www.morressier.com') THEN 'URL host -> type'
      WHEN f_host IN ('goodreads.com', 'www.goodreads.com') THEN 'URL host -> type'
      WHEN f_host IN ('picryl.com', 'www.picryl.com') THEN 'URL host -> type'
      WHEN f_src IN ('abstracts', 'abstracts with programs - geological society of america', 'academy of management proceedings', 'endocrine abstracts', 'the proceedings of the annual convention of the japanese psychological association') THEN 'source-name exact -> type'
      WHEN f_src IN ('brill’s new pauly', 'definitions', 'der neue pauly', 'encyclopédie de l’islam', 'iucn red list of threatened species', 'lexikon des gesamten buchwesens online', 'radiopaedia.org', 'religion in geschichte und gegenwart', 'springerreference', 'supplementum epigraphicum graecum', 'the shafr guide online', 'who was who', 'who\'s who') THEN 'source-name exact -> type'
      WHEN f_src IN ('psyctests dataset') THEN 'source-name exact -> type'
      WHEN f_src IN ('research square', 'ssrn electronic journal') THEN 'source-name exact -> type'
      WHEN f_src IN ('data in brief') THEN 'source-name exact -> type'
      WHEN f_src IN ('softwarex', 'the journal of open source software') THEN 'source-name exact -> type'
      WHEN f_src IN ('acta horticulturae', 'ecs transactions', 'iceri proceedings', 'ifac proceedings volumes', 'materials today proceedings', 'procedia engineering') THEN 'source-name exact -> type'
      WHEN f_src IN ('faculty opinions – post-publication peer review of the biomedical literature') THEN 'source-name exact -> type'
      WHEN f_src IN ('apress ebooks', 'jaypee brothers medical publishers (p) ltd. ebooks') THEN 'source-name exact -> type'
      WHEN f_src IN ('bulletin of the center for children\'s books', 'choice reviews online') THEN 'source-name exact -> type'
      WHEN f_src IN ('electronic enlightenment scholarly edition of correspondence') THEN 'source-name exact -> type'
      WHEN f_src IN ('national bureau of economic research') THEN 'source-name exact -> type'
      WHEN f_src IN ('synfacts') THEN 'source-name exact -> type'
      WHEN f_sc LIKE '%datasets%' THEN 'source substring (hard) -> type'
      WHEN f_sc LIKE '%web of conferences%' THEN 'source substring (hard) -> type'
      WHEN f_sc LIKE '%rxiv%' THEN 'source substring (hard) -> type'
      WHEN f_sc LIKE '%preprint%' THEN 'source substring (hard) -> type'
      WHEN f_sc LIKE '%dictionary%' THEN 'source substring (hard) -> type'
      WHEN f_sc LIKE '%encyclopedia%' THEN 'source substring (hard) -> type'
      WHEN f_sc LIKE '%lexicon%' THEN 'source substring (hard) -> type'
      WHEN f_sc LIKE '%meeting abstracts%' THEN 'source substring (hard) -> type'
      WHEN f_src IN ('e3s web of conferences', 'lecture notes on data engineering and communications technologies', 'procedia - social and behavioral sciences') THEN '#547 single-type src (hard)'
      WHEN f_src IN ('european urology supplements') THEN '#547 single-type src (hard)'
      WHEN f_src IN ('gisaid') THEN '#547 single-type src (hard)'
      WHEN (f_src LIKE '%encode%' OR f_cont LIKE '%encode%') THEN '#547 single-type src (hard)'
      WHEN (f_src LIKE '%spie proceedings%' OR f_cont LIKE '%spie proceedings%') THEN '#547 single-type src (hard)'
      WHEN (f_src LIKE '%worldwide protein data bank%' OR f_cont LIKE '%worldwide protein data bank%') THEN '#547 single-type src (hard)'
      WHEN (f_src LIKE '%sae technical paper series%' OR f_cont LIKE '%sae technical paper series%') THEN '#547 single-type src (hard)'
      WHEN (f_src LIKE '%advances in social science, education and humanities research%' OR f_cont LIKE '%advances in social science, education and humanities research%') THEN '#547 single-type src (hard)'
      WHEN (f_src LIKE '%conference on lasers and electro-optics%' OR f_cont LIKE '%conference on lasers and electro-optics%') THEN '#547 single-type src (hard)'
      WHEN (f_src LIKE '%ifmbe proceedings%' OR f_cont LIKE '%ifmbe proceedings%') THEN '#547 single-type src (hard)'
      WHEN (f_src LIKE '%morphosource%' OR f_cont LIKE '%morphosource%') THEN '#547 single-type src (hard)'
      WHEN (f_src LIKE '%sgem international multidisciplinary scientific geoconference%' OR f_cont LIKE '%sgem international multidisciplinary scientific geoconference%') THEN '#547 single-type src (hard)'
      WHEN f_doi LIKE '%meetingabstracts%' OR f_doi LIKE '%meeting-abstracts%' OR f_url LIKE '%meetingabstracts%' OR f_url LIKE '%meeting-abstracts%' THEN 'K: meetingabstracts doi/url'
      WHEN f_title LIKE 'editorial board%' THEN 'K: title editorial-board -> para'
      WHEN f_title LIKE 'front matter%' THEN 'K: title front-matter -> para'
      WHEN (f_title LIKE 'preface%' OR f_title LIKE 'appendix%' OR f_title LIKE 'proofs of%') AND (f_raw IN ('book-chapter','book-part','chapter','book-section') OR f_crtype IN ('book-chapter','monograph','edited-book')) THEN 'K: book preface/appendix -> para'
      WHEN array_contains(f_urltok, 'referenceworkentry') THEN 'URL path token -> type'
      WHEN array_contains(f_urltok, 'meetingabstracts') THEN 'URL path token -> type'
      WHEN (array_contains(f_urltok, 'thesis') OR array_contains(f_urltok, 'theses') OR array_contains(f_urltok, 'dissertations')) AND f_crtype = '' AND f_srctype <> 'journal' THEN 'K: url thesis-path -> dissertation'
      WHEN k_confabs THEN 'key:citation_conference loc/date'
      WHEN k_confpap THEN 'key:citation_conference id/series'
      WHEN array_contains(f_dc, 'book-review') THEN 'dc.type value -> type'
      WHEN array_contains(f_dc, 'bookreview') THEN 'dc.type value -> type'
      WHEN array_contains(f_dc, 'book reviews') THEN 'dc.type value -> type'
      WHEN array_contains(f_dc, 'book review') THEN 'dc.type value -> type'
      WHEN array_contains(f_dc, 'reseñas') THEN 'dc.type value -> type'
      WHEN array_contains(f_dc, 'thesis') THEN 'dc.type value -> type'
      WHEN array_contains(f_dc, 'dissertação') THEN 'dc.type value -> type'
      WHEN array_contains(f_dc, 'doctoral dissertation') THEN 'dc.type value -> type'
      WHEN array_contains(f_dc, 'pg_thesis') THEN 'dc.type value -> type'
      WHEN array_contains(f_dc, 'editorial') THEN 'dc.type value -> type'
      WHEN array_contains(f_dc, 'editorialnotes') THEN 'dc.type value -> type'
      WHEN array_contains(f_dc, 'article-commentary') THEN 'dc.type value -> type'
      WHEN array_contains(f_dc, 'meeting-report') THEN 'dc.type value -> type'
      WHEN array_contains(f_dc, 'congress-abstract') THEN 'dc.type value -> type'
      WHEN array_contains(f_dc, 'oxan-executive-summary') THEN 'dc.type value -> type'
      WHEN array_contains(f_dc, 'news') THEN 'dc.type value -> type'
      WHEN array_contains(f_dc, 'chapter') THEN 'dc.type value -> type'
      WHEN f_ptl LIKE 'reply%' THEN 'page-title ^reply -> letter'
      WHEN (f_title LIKE 'supplementary%' OR f_title LIKE 'supplemental%' OR f_title LIKE 'figure from%') OR (f_title LIKE '%supplementary figure%' OR f_title LIKE '%supplementary table%' OR f_title LIKE '%supplemental material%' OR f_title LIKE '%figure from%') THEN 'title: supplementary-materials'
      WHEN (f_title LIKE 'table of contents%' OR f_title LIKE 'contents%' OR f_title LIKE 'front matter%' OR f_title LIKE 'back matter%' OR f_title LIKE 'frontmatter%' OR f_title LIKE 'front cover%' OR f_title LIKE 'editorial board%' OR f_title LIKE 'subject index%' OR f_title LIKE 'author index%' OR f_title LIKE 'name index%' OR f_title LIKE 'list of figures%' OR f_title LIKE 'list of tables%' OR f_title LIKE 'list of contributors%' OR f_title LIKE 'list of abbreviations%' OR f_title LIKE 'list of illustrations%' OR f_title LIKE 'list of plates%' OR f_title LIKE 'bibliography%' OR f_title LIKE 'abbreviations%' OR f_title LIKE 'abbreviation%' OR f_title LIKE 'acknowledgment%' OR f_title LIKE 'acknowledgments%' OR f_title LIKE 'acknowledgement%' OR f_title LIKE 'acknowledgements%' OR f_title LIKE 'dedication%' OR f_title LIKE 'contributors%' OR f_title LIKE 'about the author%' OR f_title LIKE 'about the editor%' OR f_title LIKE 'copyright%' OR f_title LIKE 'title page%' OR f_title LIKE 'masthead%' OR f_title LIKE 'frontispiece%' OR f_title LIKE 'titelei%' OR f_title LIKE 'inhaltsverzeichnis%' OR f_title LIKE 'sachregister%' OR f_title LIKE 'literaturverzeichnis%' OR f_title LIKE 'inhalt%' OR f_title LIKE 'session details%' OR f_title LIKE 'forthcoming%' OR f_title LIKE 'calendar%' OR f_title LIKE 'general index%' OR f_title LIKE 'back cover%' OR f_title LIKE 'inside front cover%' OR f_title LIKE 'prelims%' OR f_title LIKE 'preliminary material%' OR f_title LIKE 'backmatter%' OR f_title LIKE 'books received%' OR f_title LIKE 'works cited%' OR f_title LIKE 'about the contributors%' OR f_title LIKE 'author biograph%' OR f_title LIKE 'expediente%' OR f_title LIKE 'table des mati%' OR f_title LIKE 'remerciements%') THEN 'title: paratext starts-lexicon'
      WHEN (f_title LIKE '%issue information%' OR f_title LIKE '%masthead%' OR f_title LIKE '%editorial board%' OR f_title LIKE '%instructions for authors%' OR f_title LIKE '%list of reviewers%' OR f_title LIKE '%acknowledgment of reviewers%' OR f_title LIKE '%acknowledgement of reviewers%' OR f_title LIKE '%cover image%' OR f_title LIKE '%information for authors%' OR f_title LIKE '%society information%' OR f_title LIKE '%information for contributors%' OR f_title LIKE '%information for readers%' OR f_title LIKE '%notes for contributors%' OR f_title LIKE '%notes on contributors%' OR f_title LIKE '%call for papers%' OR f_title LIKE '%call for submissions%' OR f_title LIKE '%call for abstracts%' OR f_title LIKE '%guide for authors%' OR f_title LIKE '%impressum%' OR f_title LIKE '%publication information%' OR f_title LIKE '%reviewer acknowledgement%' OR f_title LIKE '%additional journal content%' OR f_title LIKE '%[advertisement]%' OR f_title LIKE '%[advertisements]%') THEN 'title: paratext contains-lexicon'
      WHEN trim(f_title) RLIKE '^[a-z]+ (19|20)[0-9]{2} placement ads$' THEN 'title: paratext placement-ads (ZD23272)'
      WHEN trim(f_title) = 'notes' THEN 'title: paratext == notes'
      WHEN trim(f_title) = 'peer review statement' THEN 'K: title == peer review statement'
      WHEN (f_title LIKE 'program committee%' OR f_title LIKE 'organizing committee%' OR f_title LIKE 'workshop committee%' OR f_title LIKE 'conference committee%' OR f_title LIKE 'scientific committee%' OR f_title LIKE 'technical program committee%' OR f_title LIKE 'steering committee%') OR trim(f_title) RLIKE '^(program |organizing |scientific |technical |workshop |conference |steering )?committee(s)?( members| list(ing)?s?)?$' THEN 'K: title committee -> para (PT-3)'
      WHEN f_title LIKE 'index%' OR ((f_title LIKE 'references%' OR f_title LIKE 'list of%') AND (f_fp IN ('i','ii','iii','iv','ix','v','vi','vii','viii','x','xi','xii','xiii','xiv','xv') OR f_nrefs = 0 OR NOT f_hasabs)) THEN 'title: paratext ^index / idx+guard'
      WHEN (f_title LIKE '%python package%') THEN 'title: software-paper'
      WHEN (f_title LIKE 'din en%' OR f_title LIKE 'specification for%' OR f_title LIKE 'test method%') OR (f_title LIKE '%englische fassung%') THEN 'title: standard'
      WHEN (f_title LIKE 'encsr%') THEN 'title: dataset (deposit)'
      WHEN (f_title LIKE 'book review%' OR f_title LIKE 'review of the book%' OR f_title LIKE 'reseña del libro%') OR (f_title LIKE '% isbn%' OR f_title LIKE '%edited by%' OR f_title LIKE '%[book review%' OR f_title LIKE '%(book review%') OR array_contains(f_dc, 'book-review') OR (f_title LIKE '%pp.%' AND (f_title LIKE '%isbn%' OR f_title LIKE '%press%' OR f_title LIKE '%£%')) THEN 'title: book-review'
      WHEN (f_title LIKE 'guest editorial%' OR f_title LIKE 'editorial comment%' OR f_title LIKE 'guest editor%' OR f_title LIKE 'commentary on%' OR f_title LIKE 'message from%' OR f_title LIKE 'editorial board is%' OR f_title LIKE 'editorial:%' OR f_title LIKE 'preface:%' OR f_title LIKE 'préambule%' OR f_title LIKE 'éditorial%' OR f_title LIKE 'editors\' note%' OR f_title LIKE 'editors note%' OR f_title LIKE 'special thanks%' OR f_title LIKE 'nota de la directora%' OR f_title LIKE 'note from the editor%' OR f_title LIKE 'interview with%' OR f_title LIKE 'interview:%' OR f_title LIKE 'entrevista%') OR (f_title LIKE '%from the editor%' OR f_title LIKE '%special issue on%' OR f_title LIKE '%to the special issue%' OR f_title LIKE '%commentary:%') OR (f_title LIKE 'editorial%' AND f_title NOT LIKE '%board%') THEN 'title: editorial'
      WHEN trim(f_title) IN ('presentación', 'presentación.', 'presentacion', 'presentacion.', 'presentació', 'presentació.') AND f_raw NOT LIKE '%book%' AND f_raw NOT LIKE '%chapter%' AND f_raw NOT LIKE '%capítulo%' AND f_raw NOT LIKE '%libro%' AND f_raw NOT LIKE '%conference%' AND f_raw NOT LIKE '%thesis%' AND f_raw NOT LIKE '%presentation%' AND f_raw NOT LIKE '%ponencia%' AND f_raw NOT LIKE 'tesis%' AND f_raw NOT IN ('dataset','database') AND f_crtype NOT IN ('book-chapter','monograph','edited-book') THEN 'title: presentación -> editorial (ZD5175)'
      WHEN (f_title LIKE 'letter to the%' OR f_title LIKE 'reply to%' OR f_title LIKE 'in reply%' OR f_title LIKE 'reader response%' OR f_title LIKE 'comments on the article%') OR (f_title LIKE '%to the editor%' OR f_title LIKE '%authors\' reply%' OR f_title LIKE '%reply to comment%') OR ((f_title LIKE 'reply%' OR f_title LIKE 'comment on%') AND f_single) OR f_title LIKE 'correspondence%' THEN 'title: letter'
      WHEN (f_title LIKE '%narrative review%' OR f_title LIKE '%mini-review%' OR f_title LIKE '%meta-analysis of%') THEN 'title: review'
      WHEN (f_title LIKE 'libguides%' OR f_title LIKE 'all guides%' OR f_title LIKE 'research guides%') THEN 'K: title libguides'
      WHEN (f_title LIKE 're:%' OR f_title LIKE 'the authors reply%' OR f_title LIKE 'comment on:%') THEN 'K: title letter starts'
      WHEN f_title LIKE 'discussion of%' THEN 'K: title discussion-of -> editorial'
      WHEN f_title LIKE 'data for %' THEN 'K: title data-for -> dataset'
      WHEN f_title LIKE '%systematic literature review%' AND NOT (f_title LIKE '%case report%' OR f_title LIKE '%case study%') THEN 'title: systematic-lit-review (Jason guard)'
      WHEN (f_title LIKE '%in memoriam%' OR f_title LIKE '%autograph letter%' OR f_title LIKE '%obituary%') THEN 'title: other'
      WHEN f_title LIKE 'abstract%' THEN 'title: conference-abstract'
      WHEN (f_src LIKE '%abstract%' OR f_cont LIKE '%abstract%') AND (f_single OR (f_nrefs = 0 AND f_hasabs)) THEN 'struct: conf-abstract (abstracts venue)'
      WHEN f_src LIKE '%supplement%' AND f_single AND f_nrefs = 0 THEN 'struct: conf-abstract (suppl+single)'
      WHEN f_issue LIKE '%suppl%' AND f_single THEN 'struct: conf-abstract (suppl issue)'
      WHEN f_raw = 'journal-article' AND f_nrefs = 0 AND f_single AND (f_issue RLIKE '^s[0-9]' OR f_issue RLIKE '^[0-9]+s$') THEN 'struct: conf-abstract (numeric suppl)'
      WHEN (f_abs LIKE '%abstracts of presentations%' OR f_abs LIKE '%searchable abstracts%') THEN 'abs: conf-abstract phrases'
      WHEN ltrim(f_abs) LIKE 'reviewed by%' THEN 'abs: book-review (reviewed by)'
      WHEN (f_abs LIKE '%this data article%') THEN 'abs: data-paper (this data article)'
      WHEN (f_abs LIKE '%this editorial%' OR f_abs LIKE '%in this editorial%') THEN 'abs: editorial (this editorial)'
      WHEN f_src IN ('communications in computer and information science', 'energy procedia', 'lecture notes in civil engineering', 'lecture notes in computer science', 'procedia computer science') AND (f_nrefs = 0 AND f_single AND f_hasabs) THEN '#547 single-type src (guarded)'
      WHEN f_src IN ('communications in computer and information science', 'energy procedia', 'lecture notes in civil engineering', 'lecture notes in computer science', 'procedia computer science') THEN '#547 single-type src (guarded)'
      WHEN f_src IN ('scientific data') THEN '#547 single-type src (guarded)'
      WHEN (f_src LIKE '%journal of physics: conference series%' OR f_cont LIKE '%journal of physics: conference series%') AND (f_nrefs = 0 AND f_single AND f_hasabs) THEN '#547 single-type src (guarded)'
      WHEN (f_src LIKE '%journal of physics: conference series%' OR f_cont LIKE '%journal of physics: conference series%') THEN '#547 single-type src (guarded)'
      WHEN f_title RLIKE '^[a-z]{1,3}-?[0-9]{2,5}[.:\\s\\p{Z}]' AND f_nrefs = 0 AND f_raw NOT IN ('dataset','database') AND ((f_title_raw RLIKE '^[A-Z]{1,3}-?[0-9]{2,5}[.:\\s\\p{Z}]' AND NOT f_title_raw RLIKE '^(CD|IL|HSP|FGF|SOX|MMP|MIR|PSD|BCL|TNF|HLA|EGF|TGF)-?[0-9]') OR f_title_raw RLIKE '^(Mo|Tu|We|Th|Fr|Sa|Su|Mon|Tue|Wed|Thu|Fri|Sat|Sun)-?[0-9]{3,5}[.:\\s\\p{Z}]' OR f_title_raw RLIKE '^[A-Za-z]{1,3}-?[0-9]{4,5}[.:\\s\\p{Z}]') THEN 'guard: conf-abstract (title code)'
      WHEN f_title LIKE '%systematic review%' AND f_nrefs > 0 THEN 'guard: review (systematic+refs)'
      WHEN f_oatype = 'review' AND f_nrefs >= 25 AND f_hasabs THEN 'oa_type=review (+refs+abstract)'
      WHEN f_sc LIKE '%conference%' AND (f_nrefs = 0 AND f_single AND f_hasabs) THEN 'source substring (conf, guarded)'
      WHEN f_sc LIKE '%conference%' THEN 'source substring (conf, guarded)'
      WHEN f_sc LIKE '%symposium%' AND (f_nrefs = 0 AND f_single AND f_hasabs) THEN 'source substring (conf, guarded)'
      WHEN f_sc LIKE '%symposium%' THEN 'source substring (conf, guarded)'
      WHEN f_sc LIKE '%workshop%' AND (f_nrefs = 0 AND f_single AND f_hasabs) THEN 'source substring (conf, guarded)'
      WHEN f_sc LIKE '%workshop%' THEN 'source substring (conf, guarded)'
      WHEN f_raw = 'proceedings-article' AND (f_nrefs = 0 AND f_single AND f_hasabs) THEN 'raw=proceedings-article split'
      WHEN f_raw = 'proceedings-article' THEN 'raw=proceedings-article split'
      WHEN f_raw = 'proceedings' AND f_crtype = '' AND f_title NOT LIKE 'proceedings%' AND (f_nrefs = 0 AND f_single AND f_hasabs) THEN 'K: raw=proceedings repo-shaped -> conf-paper'
      WHEN f_raw = 'proceedings' AND f_crtype = '' AND f_title NOT LIKE 'proceedings%' THEN 'K: raw=proceedings repo-shaped -> conf-paper'
      WHEN f_doi LIKE '10.1596/1813-9450-%' THEN 'doi: WB PRWP series -> report (ZD23253)'
      WHEN f_crtype = 'journal-issue' THEN 'cr=journal-issue -> paratext'
      WHEN f_crtype IN ('edited-book','monograph') THEN 'cr=edited-book/monograph -> book'
      WHEN f_raw = 'reference-entry' THEN 'raw=reference-entry'
      WHEN f_raw = 'dissertation' THEN 'raw=dissertation'
      WHEN f_nrefs >= 20 AND (rtrim(f_title, ' .') LIKE '%a review' OR rtrim(f_title, ' .') LIKE '%a literature review' OR f_title LIKE '%scientometric review%') THEN 'K: title ends \'a review\''
      WHEN f_title LIKE '%a meta-analysis%' AND f_nrefs >= 20 THEN 'K: title meta-analysis'
      WHEN f_raw LIKE '%eu-repo/semantics/%' AND trim(f_raw) LIKE '%/conferenceobject' THEN 'K: raw eu-repo/semantics map'
      WHEN f_raw LIKE '%eu-repo/semantics/%' AND trim(f_raw) LIKE '%/bookpart' THEN 'K: raw eu-repo/semantics map'
      WHEN f_raw LIKE '%eu-repo/semantics/%' AND trim(f_raw) LIKE '%/doctoralthesis' THEN 'K: raw eu-repo/semantics map'
      WHEN f_raw LIKE '%eu-repo/semantics/%' AND trim(f_raw) LIKE '%/masterthesis' THEN 'K: raw eu-repo/semantics map'
      WHEN f_raw LIKE '%eu-repo/semantics/%' AND trim(f_raw) LIKE '%/article' THEN 'K: raw eu-repo/semantics map'
      WHEN f_raw LIKE '%eu-repo/semantics/%' AND trim(f_raw) LIKE '%/report' THEN 'K: raw eu-repo/semantics map'
      WHEN f_raw LIKE '%eu-repo/semantics/%' AND trim(f_raw) LIKE '%/other' THEN 'K: raw eu-repo/semantics map'
      WHEN f_raw LIKE '%thesis%' THEN 'K: raw thesis-family'
      WHEN f_raw LIKE '%väitöskirja%' THEN 'K: raw väitöskirja -> dissertation'
      WHEN f_raw LIKE '%hochschulschrift%' THEN 'K2: raw hochschulschrift'
      WHEN (f_raw LIKE 'tesis%' OR f_raw LIKE '%bakalářská práce%') THEN 'K2: raw thesis vocab (multiling)'
      WHEN f_raw LIKE '%final year project%' THEN 'K: raw final-year-project -> report'
      WHEN f_rawnorm IN ('chapter','bookpart') THEN 'K: raw chapter/bookpart'
      WHEN f_rawnorm LIKE '%conferencepaper' THEN 'K: raw conferencepaper'
      WHEN f_rawnorm = 'researchreport' THEN 'K: raw research-report'
      WHEN f_raw = 'figure' THEN 'K: raw figure -> supp-mat'
      WHEN f_rawnorm = 'software,multimedia' THEN 'K: raw software,multimedia -> other'
      WHEN f_raw = 'software' THEN 'K: raw software -> software'
      WHEN f_raw LIKE '%printed serial%' THEN 'K: raw printed-serial -> other'
      WHEN f_rawnorm IN ('image','physicalobject') THEN 'K: raw image/physobj -> other'
      WHEN f_rawnorm IN ('audiovisual','sound') THEN 'K2: raw audiovisual/sound -> other'
      WHEN (f_raw LIKE '%monograf%' OR f_raw LIKE '%monograph%') THEN 'K2: raw monograf -> book'
      WHEN f_rawnorm LIKE '%book' AND f_raw NOT IN ('book','edited-book','monograph','book-set') THEN 'K: raw ends-in book'
      WHEN f_raw LIKE '%preprint%' AND NOT (f_raw LIKE '%eu-repo%' AND NOT trim(f_raw) LIKE '%/preprint') AND NOT (f_srctype = 'journal' AND NOT (f_src LIKE '%rxiv%' OR f_src LIKE '%preprint%' OR f_src LIKE '%repec%' OR f_src LIKE '%ssrn%' OR f_src LIKE '%zenodo%' OR f_src LIKE '%research square%' OR f_src LIKE '%osf%')) AND NOT f_hasjournal THEN 'K: raw preprint (server guard)'
      WHEN (trim(f_title) RLIKE '[0-9] ?(pp|pág(s|inas)?)\\.?$' OR f_title LIKE '%(isbn%') AND f_raw NOT IN ('book-chapter','book-part','book-section','book','edited-book','monograph','book-set','report','dataset','database','proceedings','posted-content') THEN 'title: book-review citation-shape (ZD5175)'
      WHEN f_raw IN ('book-chapter','book-part') THEN 'default: raw=book-chapter/part'
      WHEN f_raw = 'book-section' THEN 'default: raw=book-section -> ref'
      WHEN f_raw IN ('book','edited-book','monograph','book-set') THEN 'default: raw=book-family -> book'
      WHEN f_raw = 'report' THEN 'default: raw=report -> report'
      WHEN f_raw = 'posted-content' THEN 'default: raw=posted-content -> other'
      WHEN f_raw IN ('dataset','database') THEN 'default: raw=dataset/database -> ds'
      WHEN f_raw = 'proceedings' THEN 'default: raw=proceedings -> para'
      WHEN f_raw = 'other' THEN 'default: raw=other -> other'
      ELSE 'default: -> article' END AS cascade_rule
  FROM feat2
  QUALIFY row_number() OVER (PARTITION BY work_id ORDER BY cascade_type, cascade_rule) = 1
),
dict_map AS (
  SELECT * FROM (VALUES
    ('repo', 'acceptedversion', 'article'),
    ('repo', 'article', 'article'),
    ('repo', 'article / letter to editor', 'article'),
    ('repo', 'artigo de jornal', 'article'),
    ('repo', 'award/grant', 'award'),
    ('repo', 'bachelor thesis', 'dissertation'),
    ('repo', 'bachelorthesis', 'dissertation'),
    ('repo', 'book', 'book'),
    ('repo', 'book article', 'book-chapter'),
    ('repo', 'book part', 'book-chapter'),
    ('repo', 'book sections', 'book-chapter'),
    ('repo', 'bookpart', 'book-chapter'),
    ('repo', 'books', 'book'),
    ('repo', 'chapter, part of book', 'book-chapter'),
    ('repo', 'chemical structures', 'other'),
    ('repo', 'conference paper', 'article'),
    ('repo', 'conference papers', 'article'),
    ('repo', 'conferencecontribution', 'article'),
    ('repo', 'conferenceitem', 'article'),
    ('repo', 'conferenceobject', 'article'),
    ('repo', 'conferencepaper', 'article'),
    ('repo', 'conferenceposter', 'article'),
    ('repo', 'conferenceproceedings', 'article'),
    ('repo', 'contributiontoperiodical', 'article'),
    ('repo', 'creative project', 'other'),
    ('repo', 'dataset', 'dataset'),
    ('repo', 'dataset/mass spectrometry', 'dataset'),
    ('repo', 'diplomová práce', 'dissertation'),
    ('repo', 'dissertation', 'dissertation'),
    ('repo', 'dissertation-reproduction (electronic)', 'dissertation'),
    ('repo', 'dissertação', 'dissertation'),
    ('repo', 'doc-type:article', 'article'),
    ('repo', 'doc-type:bookpart', 'book-chapter'),
    ('repo', 'doc-type:doctoralthesis', 'dissertation'),
    ('repo', 'doctor of philosophy', 'dissertation'),
    ('repo', 'doctoral thesis', 'dissertation'),
    ('repo', 'doctoral_dissertation', 'dissertation'),
    ('repo', 'doctoralthesis', 'dissertation'),
    ('repo', 'electronic dissertation', 'dissertation'),
    ('repo', 'hochschulschrift', 'dissertation'),
    ('repo', 'http://purl.org/coar/resource_type/c_18gh', 'report'),
    ('repo', 'http://purl.org/coar/resource_type/c_18ws', 'report'),
    ('repo', 'http://purl.org/coar/resource_type/c_2f33', 'book'),
    ('repo', 'http://purl.org/coar/resource_type/c_3248', 'book-chapter'),
    ('repo', 'http://purl.org/coar/resource_type/c_46ec', 'dissertation'),
    ('repo', 'http://purl.org/coar/resource_type/c_5794', 'conference-paper'),
    ('repo', 'http://purl.org/coar/resource_type/c_8042', 'report'),
    ('repo', 'http://purl.org/coar/resource_type/c_816b', 'preprint'),
    ('repo', 'http://purl.org/coar/resource_type/c_ba08', 'review'),
    ('repo', 'http://purl.org/coar/resource_type/c_beb9', 'dataset'),
    ('repo', 'http://purl.org/coar/resource_type/c_db06', 'dissertation'),
    ('repo', 'http://purl.org/coar/resource_type/c_dcae04bc', 'review'),
    ('repo', 'http://purl.org/coar/resource_type/c_efa0', 'conference-abstract'),
    ('repo', 'image', 'other'),
    ('repo', 'info:ulb-repo/semantics/openurl/article', 'article'),
    ('repo', 'inproceedings', 'article'),
    ('repo', 'journal article', 'article'),
    ('repo', 'journal articles', 'article'),
    ('repo', 'journal contribution', 'article'),
    ('repo', 'konferenzschrift', 'article'),
    ('repo', 'learning object', 'other'),
    ('repo', 'lecture', 'other'),
    ('repo', 'letter', 'article'),
    ('repo', 'libros', 'book'),
    ('repo', 'manuscript', 'article'),
    ('repo', 'master thesis', 'dissertation'),
    ('repo', 'masters paper', 'dissertation'),
    ('repo', 'masters thesis', 'dissertation'),
    ('repo', 'masterthesis', 'dissertation'),
    ('repo', 'monografische reihe', 'book'),
    ('repo', 'monograph', 'book'),
    ('repo', 'null', 'other'),
    ('repo', 'other', 'other'),
    ('repo', 'part of book or chapter of book', 'book-chapter'),
    ('repo', 'patent', 'other'),
    ('repo', 'peer reviewed', 'article'),
    ('repo', 'peer-review', 'peer-review'),
    ('repo', 'peerreviewed', 'article'),
    ('repo', 'phd', 'dissertation'),
    ('repo', 'phdthesis', 'dissertation'),
    ('repo', 'preprint', 'preprint'),
    ('repo', 'preprints, working papers, ...', 'preprint'),
    ('repo', 'presentation', 'other'),
    ('repo', 'publishedversion', 'article'),
    ('repo', 'report', 'report'),
    ('repo', 'reportpart', 'report'),
    ('repo', 'reports', 'report'),
    ('repo', 'research data', 'dataset'),
    ('repo', 'review', 'review'),
    ('repo', 'review article', 'review'),
    ('repo', 'software', 'software'),
    ('repo', 'submittedversion', 'article'),
    ('repo', 'technical documentation', 'report'),
    ('repo', 'technical report', 'report'),
    ('repo', 'tesi doctoral', 'dissertation'),
    ('repo', 'text', 'article'),
    ('repo', 'text (article)', 'article'),
    ('repo', 'theses', 'dissertation'),
    ('repo', 'thesis', 'dissertation'),
    ('repo', 'thesis or dissertation', 'dissertation'),
    ('repo', 'thesis-reproduction (electronic)', 'dissertation'),
    ('repo', 'thèse', 'dissertation'),
    ('repo', 'undergraduate senior honors thesis', 'dissertation'),
    ('repo', 'volume', 'book'),
    ('repo', 'vor', 'article'),
    ('repo', 'working paper', 'report'),
    ('repo', 'working paper (numbered series)', 'report'),
    ('repo', 'working papers', 'report'),
    ('repo', 'workingpaper', 'report'),
    ('repo', 'zeitschrift', 'article'),
    -- phase-1 additions (type-regression fix 2026-08-21): 172 dict-agrees keys,
    -- generated from RAW_TYPE_RANKING_MAP vs the 8863 historical oracle; additive only.
    ('repo', '  image', 'other'),
    ('repo', '  info:eu-repo/semantics/book', 'book'),
    ('repo', '  info:eu-repo/semantics/book ', 'book'),
    ('repo', ' image', 'other'),
    ('repo', ' image ', 'other'),
    ('repo', ' info:eu-repo/semantics/book', 'book'),
    ('repo', ' preprint ', 'preprint'),
    ('repo', '/info:eu-repo/semantics/other', 'other'),
    ('repo', '<dc:type/><dc:type/><dc:type/><dc:type>info:eu-repo/semantics/review', 'review'),
    ('repo', '<dc:type/><dc:type>info:eu-repo/semantics/review', 'review'),
    ('repo', '<dc:type>info:eu-repo/semantics/book', 'book'),
    ('repo', '<dc:type>info:eu-repo/semantics/conferenceobject', 'conference-paper'),
    ('repo', '<dc:type>info:eu-repo/semantics/doctoralthesis', 'dissertation'),
    ('repo', '<dc:type>info:eu-repo/semantics/masterthesis', 'dissertation'),
    ('repo', '\\n                                info:eu-repo/semantics/doctoralthesis\\n                            ', 'dissertation'),
    ('repo', '\\n            \\n                issue\\n            \\n        ', 'paratext'),
    ('repo', 'academic exercise', 'dissertation'),
    ('repo', 'afisz/plakat', 'conference-abstract'),
    ('repo', 'ancientbook', 'book'),
    ('repo', 'artículo de revisión ', 'review'),
    ('repo', 'baccalaureus work - undergraduate programme', 'dissertation'),
    ('repo', 'book ', 'book'),
    ('repo', 'book review ', 'book-review'),
    ('repo', 'book review  ', 'book-review'),
    ('repo', 'book section / proceedings', 'book-chapter'),
    ('repo', 'book/monograph/conference proceedings', 'book'),
    ('repo', 'book:ebooksection', 'book-chapter'),
    ('repo', 'books ', 'book'),
    ('repo', 'buch / monografie', 'book'),
    ('repo', 'conference article (ca)', 'conference-paper'),
    ('repo', 'conference paper, poster, etc.', 'conference-paper'),
    ('repo', 'conference-meeting part', 'conference-paper'),
    ('repo', 'conferenceobject ', 'conference-paper'),
    ('repo', 'contribution ã  ouvrage collectif (book chapter) ', 'book-chapter'),
    ('repo', 'correspondence ', 'letter'),
    ('repo', 'correspondencia', 'letter'),
    ('repo', 'dataset ', 'dataset'),
    ('repo', 'dc.type\\tinfo:eu-repo/semantics/book', 'book'),
    ('repo', 'diploma_thesis', 'dissertation'),
    ('repo', 'discussion paper ', 'report'),
    ('repo', 'diss.', 'dissertation'),
    ('repo', 'diss.(doctoral)', 'dissertation'),
    ('repo', 'doctor of sciences', 'dissertation'),
    ('repo', 'doctoral thesis ', 'dissertation'),
    ('repo', 'doctoral thesis  ', 'dissertation'),
    ('repo', 'doctoral thesis   ', 'dissertation'),
    ('repo', 'doctoral thesis    ', 'dissertation'),
    ('repo', 'doctoral thesis     ', 'dissertation'),
    ('repo', 'doctoral thesis (article-based)', 'dissertation'),
    ('repo', 'doctoralthesis ', 'dissertation'),
    ('repo', 'doctoralthesisexposure', 'dissertation'),
    ('repo', 'documento relativo ad un convegno o altro evento', 'conference-paper'),
    ('repo', 'doktorarbeit', 'dissertation'),
    ('repo', 'doktorat', 'dissertation'),
    ('repo', 'editorial ', 'editorial'),
    ('repo', 'factsheet ', 'report'),
    ('repo', 'fi=d4 julkaistu kehittämis- tai tutkimusraportti taikka -selvitys|sv=d4 publicerad utvecklings- eller forskningsrapport samt utredningar|en=d4 published development or research report or study|', 'report'),
    ('repo', 'fi=väitöskirja | en=doctoral dissertation|', 'dissertation'),
    ('repo', 'http://purl.org/eprint/type/thesis', 'dissertation'),
    ('repo', 'https://purl.org/coar/resource_type/c_2f33', 'book'),
    ('repo', 'https://purl.org/coar/resource_type/c_8042', 'report'),
    ('repo', 'https://purl.org/coar/resource_type/c_db06', 'dissertation'),
    ('repo', 'https://vocabularies.coar-repositories.org/resource_types/c_3248/', 'book-chapter'),
    ('repo', 'image ', 'other'),
    ('repo', 'image  ', 'other'),
    ('repo', 'industrystandard', 'standard'),
    ('repo', 'info:ar-repo/semantics/libro ', 'book'),
    ('repo', 'info:eu-repo/semantics/bachelor thesis', 'dissertation'),
    ('repo', 'info:eu-repo/semantics/bachelorthesis ', 'dissertation'),
    ('repo', 'info:eu-repo/semantics/book ', 'book'),
    ('repo', 'info:eu-repo/semantics/book part', 'book-chapter'),
    ('repo', 'info:eu-repo/semantics/book review', 'book-review'),
    ('repo', 'info:eu-repo/semantics/bookinfo:eu-repo/semantics/bookpart', 'book-chapter'),
    ('repo', 'info:eu-repo/semantics/books', 'book'),
    ('repo', 'info:eu-repo/semantics/chapter', 'book-chapter'),
    ('repo', 'info:eu-repo/semantics/conference proceedings', 'conference-paper'),
    ('repo', 'info:eu-repo/semantics/conference_object', 'conference-paper'),
    ('repo', 'info:eu-repo/semantics/conferenceitem', 'conference-paper'),
    ('repo', 'info:eu-repo/semantics/conferenceobjectinfo:eu-repo/semantics/other', 'other'),
    ('repo', 'info:eu-repo/semantics/conferencia', 'conference-paper'),
    ('repo', 'info:eu-repo/semantics/corrigendum', 'erratum'),
    ('repo', 'info:eu-repo/semantics/dissertation', 'dissertation'),
    ('repo', 'info:eu-repo/semantics/doctoral thesis', 'dissertation'),
    ('repo', 'info:eu-repo/semantics/editorial', 'editorial'),
    ('repo', 'info:eu-repo/semantics/előadáskivonat', 'conference-abstract'),
    ('repo', 'info:eu-repo/semantics/encyclopedia_article', 'reference-entry'),
    ('repo', 'info:eu-repo/semantics/image', 'other'),
    ('repo', 'info:eu-repo/semantics/info:eu-repo/semantics/doctoralthesis', 'dissertation'),
    ('repo', 'info:eu-repo/semantics/livro', 'book'),
    ('repo', 'info:eu-repo/semantics/master\'s thesis', 'dissertation'),
    ('repo', 'info:eu-repo/semantics/monograph', 'book'),
    ('repo', 'info:eu-repo/semantics/otherinfo:eu-repo/semantics/other', 'other'),
    ('repo', 'info:eu-repo/semantics/patent ', 'other'),
    ('repo', 'info:eu-repo/semantics/proceedings', 'conference-paper'),
    ('repo', 'info:eu-repo/semantics/publishedversion|info:eu-repo/semantics/masterthesis', 'dissertation'),
    ('repo', 'info:eu-repo/semantics/referenceentry', 'reference-entry'),
    ('repo', 'info:eu-repo/semantics/ressenya', 'book-review'),
    ('repo', 'info:eu-repo/semantics/review article', 'review'),
    ('repo', 'info:eu-repo/semantics/studythesis', 'dissertation'),
    ('repo', 'info:eu-repo/semantics/tanulmány, értekezés', 'dissertation'),
    ('repo', 'info:eu-repo/semantics/technical report', 'report'),
    ('repo', 'info:eu-repo/semantics/tesis de doctorado', 'dissertation'),
    ('repo', 'info:eu-repo/semantics/tesis de maestría', 'dissertation'),
    ('repo', 'info:eu-repo/semantics/workshop', 'conference-paper'),
    ('repo', 'informe final', 'report'),
    ('repo', 'libro ', 'book'),
    ('repo', 'licentiatethesis', 'dissertation'),
    ('repo', 'ma master of arts', 'dissertation'),
    ('repo', 'master in economics', 'dissertation'),
    ('repo', 'master\'s thesis - graduate programme', 'dissertation'),
    ('repo', 'masterarbeit ', 'dissertation'),
    ('repo', 'masters by research', 'dissertation'),
    ('repo', 'md doctor of medicine', 'dissertation'),
    ('repo', 'meeting abstract ', 'conference-abstract'),
    ('repo', 'mims preprint', 'preprint'),
    ('repo', 'mémoire de master', 'dissertation'),
    ('repo', 'mémoire iufm', 'dissertation'),
    ('repo', 'norma branżowa', 'standard'),
    ('repo', 'opracowanie statystyczne', 'report'),
    ('repo', 'personal correspondence ', 'letter'),
    ('repo', 'phd-thesis', 'dissertation'),
    ('repo', 'physicsthesis', 'dissertation'),
    ('repo', 'plakaty', 'conference-abstract'),
    ('repo', 'preprint ', 'preprint'),
    ('repo', 'resenha ', 'book-review'),
    ('repo', 'reseña de libro ', 'book-review'),
    ('repo', 'review info:eu-repo/semantics/review', 'review'),
    ('repo', 'review single work', 'book-review'),
    ('repo', 'skripsi,tesis,disertasi', 'dissertation'),
    ('repo', 'software educacional', 'software'),
    ('repo', 'specialistthesis', 'dissertation'),
    ('repo', 'sprawozdanie szkolne', 'report'),
    ('repo', 'squeeze paper', 'conference-paper'),
    ('repo', 'still image; poster', 'conference-abstract'),
    ('repo', 'szöveges plakát', 'conference-abstract'),
    ('repo', 'tesi di dottorato ', 'dissertation'),
    ('repo', 'tesis (pre-grado)', 'dissertation'),
    ('repo', 'tesis doctorado', 'dissertation'),
    ('repo', 'tesis i dissertacions electròniques', 'dissertation'),
    ('repo', 'tesis i dissertacions electròniques.', 'dissertation'),
    ('repo', 'tesis maestría', 'dissertation'),
    ('repo', 'tesis magister', 'dissertation'),
    ('repo', 'tesis pre-grado', 'dissertation'),
    ('repo', 'text.article.conference.poster', 'conference-abstract'),
    ('repo', 'text.preprint', 'preprint'),
    ('repo', 'text.thesis.bachelor.m', 'dissertation'),
    ('repo', 'text.thesis.bachelor.m2', 'dissertation'),
    ('repo', 'text/thesis ', 'dissertation'),
    ('repo', 'thesis ', 'dissertation'),
    ('repo', 'thesis (dr.)', 'dissertation'),
    ('repo', 'thesis (masters)', 'dissertation'),
    ('repo', 'thã¨se ou mã©moire de l\'uqac', 'dissertation'),
    ('repo', 'thèse d’exercice', 'dissertation'),
    ('repo', 'trabajo de integración curricular', 'dissertation'),
    ('repo', 'type: info:eu-repo/semantics/doctoralthesis', 'dissertation'),
    ('repo', 'ug_thesis', 'dissertation'),
    ('repo', 'undergraduates project papers', 'dissertation'),
    ('repo', 'visokošolsko delo', 'dissertation'),
    ('repo', 'working paper ', 'report'),
    ('repo', '|info:eu-repo/semantics/book', 'book'),
    ('repo', ' http://purl.org/coar/resource_type/c_46ec', 'dissertation'),
    ('repo', 'выпускная бакалаврская работа', 'dissertation'),
    ('repo', 'докторска дисертација', 'dissertation'),
    ('repo', 'магистерская диссертация', 'dissertation'),
    ('repo', 'مؤلَّف محكَّم', 'book'),
    ('repo', '专著章节/文集论文', 'book-chapter'),
    ('repo', '博士論文 (doctoral dissertation)', 'dissertation'),
    ('repo', '学位論文', 'dissertation'),
    ('repo', '学位論文(thesis)', 'dissertation'),
    ('repo', '学士', 'dissertation'),
    ('repo', '演示报告', 'conference-abstract'),
    ('repo', '研究报告', 'report'),
    -- tier-2 additions (type-regression fix 2026-08-23): screened entries,
    -- T1 crosswalk-authoritative + T2 unanimous>=100 + T3 supermajority>=99.9% + T4 repo modal>=95%.
    ('datacite', 'film', 'other'),
    ('datacite', 'outputmanagementplan', 'other'),
    ('datacite', 'project', 'other'),
    ('repo', '/dk/atira/pure/researchoutput/researchoutputtypes/contributiontobookanthology/chapter', 'other'),
    ('repo', '/dk/atira/pure/researchoutput/researchoutputtypes/contributiontojournal/case_note', 'other'),
    ('repo', '/dk/atira/pure/researchoutput/researchoutputtypes/contributiontojournal/conferencearticle', 'other'),
    ('repo', '/dk/atira/pure/researchoutput/researchoutputtypes/contributiontojournal/editorial', 'other'),
    ('repo', '/dk/atira/pure/researchoutput/researchoutputtypes/contributiontojournal/systematicreview', 'other'),
    ('repo', '/dk/atira/pure/researchoutput/researchoutputtypes/workingpaper/preprint', 'preprint'),
    ('repo', '24 bit color or 8 bit greyscale', 'other'),
    ('repo', '35mm_slide', 'other'),
    ('repo', '[data collection]', 'dataset'),
    ('repo', 'a vertaisarvioidut tieteelliset artikkelit', 'other'),
    ('repo', 'a4 artikkeli konferenssijulkaisussa', 'other'),
    ('repo', 'abst', 'conference-abstract'),
    ('repo', 'abstract', 'conference-abstract'),
    ('repo', 'academic conference', 'conference-paper'),
    ('repo', 'academic document', 'other'),
    ('repo', 'academic record', 'other'),
    ('repo', 'academic theses', 'dissertation'),
    ('repo', 'accepted manuscript', 'other'),
    ('repo', 'accreditation document', 'other'),
    ('repo', 'accreditation to supervise research', 'dissertation'),
    ('repo', 'act', 'other'),
    ('repo', 'acta / minuta', 'other'),
    ('repo', 'actas de congreso', 'conference-paper'),
    ('repo', 'actes de colloque', 'conference-paper'),
    ('repo', 'activity', 'other'),
    ('repo', 'addresses', 'other'),
    ('repo', 'administracinis / administrative', 'other'),
    ('repo', 'administrative document', 'other'),
    ('repo', 'administrative record', 'other'),
    ('repo', 'adopted plan', 'other'),
    ('repo', 'advanced master in international and development economics', 'other'),
    ('repo', 'affiche', 'other'),
    ('repo', 'affiche scientifique', 'other'),
    ('repo', 'affisch', 'conference-abstract'),
    ('repo', 'afisz', 'other'),
    ('repo', 'agendasminutes', 'other'),
    ('repo', 'akta', 'other'),
    ('repo', 'album', 'other'),
    ('repo', 'albumin negatives; black-and-white negatives', 'other'),
    ('repo', 'allgemein', 'other'),
    ('repo', 'altres', 'other'),
    ('repo', 'am', 'other'),
    ('repo', 'anais de evento', 'other'),
    ('repo', 'anais e proceedings de eventos', 'other'),
    ('repo', 'analista de sistemas', 'other'),
    ('repo', 'animation', 'other'),
    ('repo', 'annotation', 'other'),
    ('repo', 'announcement', 'other'),
    ('repo', 'annual report', 'report'),
    ('repo', 'annual_report', 'other'),
    ('repo', 'anthology', 'book'),
    ('repo', 'anthologyarticle', 'book-chapter'),
    ('repo', 'ao', 'preprint'),
    ('repo', 'application/pdf', 'other'),
    ('repo', 'arbeitspapier', 'report'),
    ('repo', 'architectural drawing', 'other'),
    ('repo', 'architectural drawings (visual works)', 'other'),
    ('repo', 'architectural model', 'other'),
    ('repo', 'architecture', 'other'),
    ('repo', 'architecture and city planning', 'other'),
    ('repo', 'architecture and city planning; decorative arts, utilitarian objects and interior design', 'other'),
    ('repo', 'architecture and city planning; drawings and watercolors', 'other'),
    ('repo', 'architecture and city planning; garden and landscape', 'other'),
    ('repo', 'architecture and city planning; photographs', 'other'),
    ('repo', 'architecture and city planning; sculpture and installations', 'other'),
    ('repo', 'architecture.', 'other'),
    ('repo', 'archival material', 'other'),
    ('repo', 'archival record', 'other'),
    ('repo', 'archivos comprimidos', 'other'),
    ('repo', 'archivos de ordenador', 'software'),
    ('repo', 'archivos de video', 'other'),
    ('repo', 'archivos textuales', 'other'),
    ('repo', 'art', 'other'),
    ('repo', 'art design item', 'other'),
    ('repo', 'art object', 'other'),
    ('repo', 'art or design object', 'other'),
    ('repo', 'art/design item', 'other'),
    ('repo', 'art; poetry', 'other'),
    ('repo', 'art; short stories', 'other'),
    ('repo', 'artefact', 'other'),
    ('repo', 'article - magazine', 'other'),
    ('repo', 'article de diari', 'other'),
    ('repo', 'article de revisió', 'review'),
    ('repo', 'article de vulgarisation', 'other'),
    ('repo', 'article in academic journal', 'other'),
    ('repo', 'article in proceedings', 'other'),
    ('repo', 'article pre-print', 'other'),
    ('repo', 'article review/survey', 'other'),
    ('repo', 'article, letter', 'other'),
    ('repo', 'article/letter to editor', 'other'),
    ('repo', 'article/review', 'review'),
    ('repo', 'article; proceedings paper', 'conference-paper'),
    ('repo', 'article_in_conference_proceedings', 'conference-paper'),
    ('repo', 'articulo revisado por pares', 'other'),
    ('repo', 'artigo em livro de atas de conferência internacional', 'other'),
    ('repo', 'artigo em revista científica nacional', 'other'),
    ('repo', 'artigo na mídia', 'other'),
    ('repo', 'artikel mahasiswa', 'other'),
    ('repo', 'artikel umum', 'other'),
    ('repo', 'artwork', 'other'),
    ('repo', 'artykuł (rozdział w książce)', 'book-chapter'),
    ('repo', 'artículo de conferencia', 'conference-paper'),
    ('repo', 'artículo de periódico', 'other'),
    ('repo', 'artículo de revisión', 'review'),
    ('repo', 'artículos y capítulos', 'other'),
    ('repo', 'asignatura', 'other'),
    ('repo', 'assemblage', 'other'),
    ('repo', 'assignment', 'other'),
    ('repo', 'atlas map', 'other'),
    ('repo', 'audio', 'other'),
    ('repo', 'audio recording', 'other'),
    ('repo', 'audio/visual', 'other'),
    ('repo', 'audio/visual -- movie/animation', 'other'),
    ('repo', 'audio/visual resource', 'other'),
    ('repo', 'aufsatz in einem buch', 'book-chapter'),
    ('repo', 'authored books', 'book'),
    ('repo', 'authors certificate', 'other'),
    ('repo', 'autoreferat', 'dissertation'),
    ('repo', 'autoreferates', 'other'),
    ('repo', 'autre communication scientifique (congrès sans actes - poster - séminaire...)', 'conference-abstract'),
    ('repo', 'autre type de document', 'other'),
    ('repo', 'avhandling pro gradu', 'dissertation'),
    ('repo', 'awaiting review', 'other'),
    ('repo', 'b vertaisarvioimattomat tieteelliset kirjoitukset', 'other'),
    ('repo', 'bachelor', 'dissertation'),
    ('repo', 'bachelor dissertation', 'other'),
    ('repo', 'bachelorarbeit', 'dissertation'),
    ('repo', 'bachelorproject', 'other'),
    ('repo', 'bachelous paper', 'dissertation'),
    ('repo', 'banner', 'other'),
    ('repo', 'basis_projekt', 'other'),
    ('repo', 'beitrag im sammelband', 'book-chapter'),
    ('repo', 'beitrag in einem lehr- oder fachbuch', 'book-chapter'),
    ('repo', 'bericht, report', 'report'),
    ('repo', 'berichtsreihe', 'report'),
    ('repo', 'bibliografia', 'other'),
    ('repo', 'bibliografie', 'other'),
    ('repo', 'biblionot', 'other'),
    ('repo', 'bild', 'other'),
    ('repo', 'bild / foto', 'other'),
    ('repo', 'bio', 'other'),
    ('repo', 'biografia', 'other'),
    ('repo', 'biogram', 'other'),
    ('repo', 'biuletyn', 'other'),
    ('repo', 'black-and-white negatives', 'other'),
    ('repo', 'black-and-white photographs', 'other'),
    ('repo', 'blog', 'other'),
    ('repo', 'blog post', 'other'),
    ('repo', 'boletín', 'other'),
    ('repo', 'book chapter', 'book-chapter'),
    ('repo', 'book chapter or section', 'book-chapter'),
    ('repo', 'book chapters', 'book-chapter'),
    ('repo', 'book editorial', 'book'),
    ('repo', 'book item', 'book'),
    ('repo', 'book of condolence', 'other'),
    ('repo', 'book or report section', 'book-chapter'),
    ('repo', 'book part or chapter', 'book-chapter'),
    ('repo', 'book review', 'book-review'),
    ('repo', 'book reviews', 'book-review'),
    ('repo', 'book section', 'book-chapter'),
    ('repo', 'book section / chapter', 'other'),
    ('repo', 'book series article/chapter', 'book-chapter'),
    ('repo', 'book/report', 'report'),
    ('repo', 'book/report/proceedings', 'report'),
    ('repo', 'book_chap', 'book-chapter'),
    ('repo', 'book_chapter', 'book-chapter'),
    ('repo', 'book_contribution', 'book-chapter'),
    ('repo', 'book_phd', 'dissertation'),
    ('repo', 'book_review', 'other'),
    ('repo', 'book_section', 'book-chapter'),
    ('repo', 'bookanthology/report', 'book'),
    ('repo', 'bookcontrib', 'other'),
    ('repo', 'bookitem', 'book-chapter'),
    ('repo', 'booklet', 'other'),
    ('repo', 'bookreview', 'book-review'),
    ('repo', 'books; school yearbooks', 'other'),
    ('repo', 'booksection', 'book-chapter'),
    ('repo', 'book ', 'book'),
    ('repo', 'brevet', 'other'),
    ('repo', 'brief', 'other'),
    ('repo', 'briefcommun', 'other'),
    ('repo', 'briefing paper', 'other'),
    ('repo', 'broadside', 'other'),
    ('repo', 'brochure', 'other'),
    ('repo', 'broszura', 'other'),
    ('repo', 'buch', 'book'),
    ('repo', 'buch / sammelwerk', 'book'),
    ('repo', 'buchkapitel', 'book-chapter'),
    ('repo', 'buchkapitel / sammelwerksbeitrag', 'book-chapter'),
    ('repo', 'buku', 'book'),
    ('repo', 'bulletin or newsletter', 'other'),
    ('repo', 'bulletins', 'other'),
    ('repo', 'business correspondence', 'other'),
    ('repo', 'business records', 'other'),
    ('repo', 'c-print', 'other'),
    ('repo', 'c1 kustannettu tieteellinen erillisteos', 'other'),
    ('repo', 'c2 toimitettu kirja, kokoomateos, konferenssijulkaisu tai lehden erikoisnumero', 'other'),
    ('repo', 'cable', 'other'),
    ('repo', 'campus_syllabus', 'other'),
    ('repo', 'campusdissertation', 'dissertation'),
    ('repo', 'capitulo de libro', 'book-chapter'),
    ('repo', 'capstone', 'dissertation'),
    ('repo', 'capstone paper', 'other'),
    ('repo', 'capstone project', 'dissertation'),
    ('repo', 'capítol de llibre', 'book-chapter'),
    ('repo', 'capítulo', 'book-chapter'),
    ('repo', 'capítulo de libro', 'book-chapter'),
    ('repo', 'capítulo de livro', 'book-chapter'),
    ('repo', 'capítulo ou parte de livro', 'other'),
    ('repo', 'card', 'other'),
    ('repo', 'carta', 'letter'),
    ('repo', 'carteles', 'conference-abstract'),
    ('repo', 'carteles de teatro', 'other'),
    ('repo', 'carteles de teatro  -murcia', 'other'),
    ('repo', 'cartells polítics', 'other'),
    ('repo', 'cartographic', 'other'),
    ('repo', 'cartographic material', 'other'),
    ('repo', 'cartography', 'other'),
    ('repo', 'carvings and sculpture', 'other'),
    ('repo', 'case', 'other'),
    ('repo', 'case map', 'other'),
    ('repo', 'case_report', 'other'),
    ('repo', 'casepresentation', 'other'),
    ('repo', 'casereport', 'other'),
    ('repo', 'catalog', 'other'),
    ('repo', 'celestial atlas', 'other'),
    ('repo', 'ceramic / antiquity', 'other'),
    ('repo', 'ceramics', 'other'),
    ('repo', 'cerámica', 'other'),
    ('repo', 'chansons', 'other'),
    ('repo', 'chapel_address', 'other'),
    ('repo', 'chapitre d\'ouvrage', 'book-chapter'),
    ('repo', 'chapitre de livre', 'book-chapter'),
    ('repo', 'chapters', 'other'),
    ('repo', 'chart atlas', 'other'),
    ('repo', 'charter', 'other'),
    ('repo', 'church', 'other'),
    ('repo', 'churches (buildings)', 'other'),
    ('repo', 'cigarette card', 'other'),
    ('repo', 'circular', 'other'),
    ('repo', 'city atlas', 'other'),
    ('repo', 'civic architecture', 'other'),
    ('repo', 'clinical aphasiology paper', 'other'),
    ('repo', 'clinical trial report', 'report'),
    ('repo', 'clipping', 'other'),
    ('repo', 'code', 'other'),
    ('repo', 'collage', 'other'),
    ('repo', 'collected articles', 'other'),
    ('repo', 'collectededition', 'book'),
    ('repo', 'collective labor agreement', 'other'),
    ('repo', 'colloquium', 'other'),
    ('repo', 'color', 'other'),
    ('repo', 'color faded', 'other'),
    ('repo', 'color negatives', 'other'),
    ('repo', 'color photographs', 'other'),
    ('repo', 'color/black', 'other'),
    ('repo', 'color/black & white', 'other'),
    ('repo', 'columnreport', 'other'),
    ('repo', 'col·lecció d\'arxiu', 'other'),
    ('repo', 'commencement', 'other'),
    ('repo', 'commencement programs', 'other'),
    ('repo', 'commentary', 'editorial'),
    ('repo', 'commissioned report', 'other'),
    ('repo', 'communication', 'other'),
    ('repo', 'communication artifacts; art 2-d; print', 'other'),
    ('repo', 'communication artifacts; art 3-d; figurine', 'other'),
    ('repo', 'communication artifacts; art 3-d; sculpture', 'other'),
    ('repo', 'communication dans un congrès', 'conference-paper'),
    ('repo', 'communication dans un congrès avec actes', 'conference-paper'),
    ('repo', 'communication de conférence', 'conference-paper'),
    ('repo', 'competitive_scientific_work', 'other'),
    ('repo', 'composition', 'other'),
    ('repo', 'compte rendu de conférence', 'conference-paper'),
    ('repo', 'computer file', 'other'),
    ('repo', 'comunicacion', 'conference-paper'),
    ('repo', 'comunicació de congrés', 'conference-paper'),
    ('repo', 'comunicación', 'conference-paper'),
    ('repo', 'comunicación de congreso', 'conference-paper'),
    ('repo', 'concert program', 'other'),
    ('repo', 'concert programs', 'other'),
    ('repo', 'conf', 'conference-paper'),
    ('repo', 'conference', 'conference-paper'),
    ('repo', 'conference abstract', 'conference-abstract'),
    ('repo', 'conference article', 'conference-paper'),
    ('repo', 'conference contribution', 'conference-paper'),
    ('repo', 'conference contribution - published', 'conference-paper'),
    ('repo', 'conference contribution - unpublished', 'conference-paper'),
    ('repo', 'conference contributions - other', 'other'),
    ('repo', 'conference contributions - published', 'conference-paper'),
    ('repo', 'conference document', 'conference-paper'),
    ('repo', 'conference item', 'conference-paper'),
    ('repo', 'conference lecture', 'conference-paper'),
    ('repo', 'conference material', 'conference-paper'),
    ('repo', 'conference materials', 'conference-paper'),
    ('repo', 'conference object', 'conference-paper'),
    ('repo', 'conference or workshop', 'other'),
    ('repo', 'conference or workshop item', 'conference-paper'),
    ('repo', 'conference or workshop item - published', 'other'),
    ('repo', 'conference or workshop items', 'conference-paper'),
    ('repo', 'conference output', 'conference-paper'),
    ('repo', 'conference paper not in proceedings', 'conference-paper'),
    ('repo', 'conference paper/proceeding/abstract', 'conference-paper'),
    ('repo', 'conference papers and proceedings', 'conference-paper'),
    ('repo', 'conference papers, meetings and proceedings', 'conference-paper'),
    ('repo', 'conference poster not in proceedings', 'conference-abstract'),
    ('repo', 'conference presentation', 'conference-paper'),
    ('repo', 'conference proceeding', 'conference-paper'),
    ('repo', 'conference proceeding article', 'conference-paper'),
    ('repo', 'conference proceedings', 'conference-paper'),
    ('repo', 'conference publication', 'conference-paper'),
    ('repo', 'conference report', 'conference-paper'),
    ('repo', 'conference, symposium or workshop item', 'conference-paper'),
    ('repo', 'conference/workshop item', 'conference-paper'),
    ('repo', 'conference_abstract', 'conference-abstract'),
    ('repo', 'conference_item', 'conference-paper'),
    ('repo', 'conference_object', 'conference-paper'),
    ('repo', 'conferencepresentation', 'conference-paper'),
    ('repo', 'conferencia', 'conference-paper'),
    ('repo', 'congressional record tribute', 'other'),
    ('repo', 'cont_refjournal', 'other'),
    ('repo', 'container', 'other'),
    ('repo', 'contribució a congrés', 'conference-paper'),
    ('repo', 'contribution for newspaper or weekly magazine', 'other'),
    ('repo', 'contribution in book/report/proceedings', 'book-chapter'),
    ('repo', 'contribution to conference', 'conference-paper'),
    ('repo', 'contribution to newspaper/magazine', 'other'),
    ('repo', 'contribution to periodical', 'other'),
    ('repo', 'contribution_to_periodical', 'other'),
    ('repo', 'contributiontojournal/systematicreview', 'review'),
    ('repo', 'copyright', 'other'),
    ('repo', 'corpus', 'other'),
    ('repo', 'corr', 'other'),
    ('repo', 'correspondence', 'letter'),
    ('repo', 'corrigendum', 'erratum'),
    ('repo', 'costume', 'other'),
    ('repo', 'county atlas', 'other'),
    ('repo', 'course of lectures', 'other'),
    ('repo', 'coursecatalogs', 'other'),
    ('repo', 'coursematerial', 'other'),
    ('repo', 'courseschedules', 'other'),
    ('repo', 'coursework', 'other'),
    ('repo', 'cover', 'other'),
    ('repo', 'creación de obra artística', 'other'),
    ('repo', 'creative component', 'dissertation'),
    ('repo', 'creative project (m.a.), 3 hrs.', 'other'),
    ('repo', 'creative project (m.m.), 3 hrs.', 'other'),
    ('repo', 'creative project, 3 hrs.', 'other'),
    ('repo', 'creative project, 4 hrs.', 'other'),
    ('repo', 'creative work', 'other'),
    ('repo', 'creative_writing', 'other'),
    ('repo', 'curricula for deep higher education (master\'s degree)', 'other'),
    ('repo', 'curricula of general higher education (bachelor\'s degree)', 'other'),
    ('repo', 'curriculum', 'other'),
    ('repo', 'curso de profundizacion', 'other'),
    ('repo', 'curso de profundización', 'other'),
    ('repo', 'curso virtual', 'other'),
    ('repo', 'czasopisma', 'other'),
    ('repo', 'czasopismo', 'other'),
    ('repo', 'czasopismo elektroniczne', 'other'),
    ('repo', 'czasopismo starodruczne', 'other'),
    ('repo', 'd4 julkaistu kehittämis- tai tutkimusraportti tai -selvitys', 'report'),
    ('repo', 'd4 julkaistu kehittämis- tai tutkimusraportti taikka -selvitys', 'report'),
    ('repo', 'data', 'dataset'),
    ('repo', 'data collection', 'dataset'),
    ('repo', 'data or dataset', 'dataset'),
    ('repo', 'data set', 'dataset'),
    ('repo', 'dataset bundled publication', 'dataset'),
    ('repo', 'dataset publication series', 'dataset'),
    ('repo', 'datasets / databases', 'other'),
    ('repo', 'deb', 'other'),
    ('repo', 'dec', 'other'),
    ('repo', 'decorative arts', 'other'),
    ('repo', 'decorative arts, utilitarian objects and interior design', 'other'),
    ('repo', 'decorative arts, utilitarian objects and interior design|process and technique', 'other'),
    ('repo', 'decr', 'other'),
    ('repo', 'degree work', 'other'),
    ('repo', 'delib', 'other'),
    ('repo', 'departmental report', 'report'),
    ('repo', 'desconocido', 'other'),
    ('repo', 'design', 'other'),
    ('repo', 'designed landscapes', 'other'),
    ('repo', 'dias', 'other'),
    ('repo', 'dicent', 'other'),
    ('repo', 'digital artefact', 'other'),
    ('repo', 'digital images', 'other'),
    ('repo', 'digital or visual media', 'other'),
    ('repo', 'digital scholarly resource', 'other'),
    ('repo', 'digitalobject', 'other'),
    ('repo', 'diplom', 'dissertation'),
    ('repo', 'diplom- oder magisterarbeit', 'dissertation'),
    ('repo', 'diploma/tugas akhir', 'dissertation'),
    ('repo', 'diplomado de profundización para grado', 'dissertation'),
    ('repo', 'diplomarbete', 'other'),
    ('repo', 'diplomas', 'dissertation'),
    ('repo', 'diplomityö', 'dissertation'),
    ('repo', 'discussion paper', 'report'),
    ('repo', 'discussion/ working paper', 'other'),
    ('repo', 'disertasi', 'other'),
    ('repo', 'disertační práce', 'dissertation'),
    ('repo', 'dissertation (5 years campus access only)', 'other'),
    ('repo', 'dissertation (campus access only)', 'dissertation'),
    ('repo', 'dissertation (open access)', 'dissertation'),
    ('repo', 'dissertation (university of nottingham only)', 'dissertation'),
    ('repo', 'dissertationcoa', 'other'),
    ('repo', 'dissertationcontrolled', 'other'),
    ('repo', 'dissertationen', 'dissertation'),
    ('repo', 'dissertations', 'dissertation'),
    ('repo', 'dissertação (mestrado)', 'dissertation'),
    ('repo', 'disszertáció', 'dissertation'),
    ('repo', 'doc-type:conferenceobject', 'conference-paper'),
    ('repo', 'doc-type:other', 'other'),
    ('repo', 'doc-type:periodicalpart', 'other'),
    ('repo', 'doc-type:preprint', 'preprint'),
    ('repo', 'doc-type:report', 'report'),
    ('repo', 'doc-type:researchdata', 'dataset'),
    ('repo', 'doc-type:review', 'review'),
    ('repo', 'doc-type:workingpaper', 'report'),
    ('repo', 'doctor of clinical psychology', 'other'),
    ('repo', 'doctor of education (edd)', 'other'),
    ('repo', 'doctor of philosophy (phd)', 'dissertation'),
    ('repo', 'doctoral', 'dissertation'),
    ('repo', 'doctoral dissertation', 'dissertation'),
    ('repo', 'doctoral dissertation (article based)', 'other'),
    ('repo', 'doctoral dissertation (article-based)', 'other'),
    ('repo', 'document', 'other'),
    ('repo', 'document de travail - pré-publication', 'report'),
    ('repo', 'document from web', 'other'),
    ('repo', 'document issu d\'une conférence ou d\'un atelier', 'conference-paper'),
    ('repo', 'documentación técnica', 'other'),
    ('repo', 'documento de conferencia', 'conference-paper'),
    ('repo', 'documento de trabajo', 'report'),
    ('repo', 'documento histórico', 'other'),
    ('repo', 'documento institucional', 'other'),
    ('repo', 'documents institucionals', 'other'),
    ('repo', 'dodatek do czasopisma', 'other'),
    ('repo', 'doktorat, praca dyplomowa', 'dissertation'),
    ('repo', 'dokument elektroniczny', 'other'),
    ('repo', 'dokument ikonograficzny', 'other'),
    ('repo', 'donor report', 'other'),
    ('repo', 'drafd', 'other'),
    ('repo', 'drawing', 'other'),
    ('repo', 'drawing / identified artist', 'other'),
    ('repo', 'drawings', 'other'),
    ('repo', 'drawings and watercolors', 'other'),
    ('repo', 'drawings and watercolors; paintings', 'other'),
    ('repo', 'drawings and watercolors; sculpture and installations', 'other'),
    ('repo', 'drawings and watercolors|charcoal|periodical illustrations', 'other'),
    ('repo', 'drawings and watercolors|periodical illustrations', 'other'),
    ('repo', 'drawings, pastels & watercolors; painting', 'other'),
    ('repo', 'drawings, pastels and watercolors; painting', 'other'),
    ('repo', 'druk', 'other'),
    ('repo', 'druk muzyczny', 'other'),
    ('repo', 'druk ulotny', 'other'),
    ('repo', 'druki ulotne', 'other'),
    ('repo', 'dvd', 'other'),
    ('repo', 'dwumiesięcznik', 'other'),
    ('repo', 'dwutygodniki', 'other'),
    ('repo', 'dzienniki urzędowe', 'other'),
    ('repo', 'dźwięk', 'other'),
    ('repo', 'e-czasopismo', 'other'),
    ('repo', 'e-książka', 'book'),
    ('repo', 'easel painting / identified artist', 'other'),
    ('repo', 'easel painting / unidentified artist', 'other'),
    ('repo', 'edited book or journal volume', 'book'),
    ('repo', 'edited books', 'other'),
    ('repo', 'edited journal', 'other'),
    ('repo', 'edited scientific work', 'book'),
    ('repo', 'edited volume', 'other'),
    ('repo', 'edito', 'editorial'),
    ('repo', 'editor', 'other'),
    ('repo', 'editorial material', 'editorial'),
    ('repo', 'editorial reviewed', 'editorial'),
    ('repo', 'editorial/preface (journal)', 'other'),
    ('repo', 'editorials/short communications', 'editorial'),
    ('repo', 'ekslibris', 'other'),
    ('repo', 'elec', 'other'),
    ('repo', 'electronic educational and methodical complex', 'other'),
    ('repo', 'electronic report', 'report'),
    ('repo', 'encyclopedia_article', 'reference-entry'),
    ('repo', 'engravings', 'other'),
    ('repo', 'ensemble de données', 'other'),
    ('repo', 'entregable de proyecto', 'other'),
    ('repo', 'entrepreneurship project', 'other'),
    ('repo', 'entretiens avec les habitants', 'other'),
    ('repo', 'entry', 'other'),
    ('repo', 'entry in reference work', 'other'),
    ('repo', 'ephemera', 'other'),
    ('repo', 'erratum', 'erratum'),
    ('repo', 'esiupseerikurssin tutkielma', 'other'),
    ('repo', 'essai', 'other'),
    ('repo', 'etacc', 'other'),
    ('repo', 'etd', 'dissertation'),
    ('repo', 'eu commission - brochure', 'other'),
    ('repo', 'eu commission - com document', 'report'),
    ('repo', 'eu commission - press notice', 'other'),
    ('repo', 'eu commission - sec document', 'other'),
    ('repo', 'eu commission - working document', 'report'),
    ('repo', 'eu council of the eu document', 'report'),
    ('repo', 'eu economic and social committee', 'other'),
    ('repo', 'eu european council', 'other'),
    ('repo', 'eu european parliament document', 'report'),
    ('repo', 'eu other', 'other'),
    ('repo', 'eu related', 'other'),
    ('repo', 'eu speech', 'other'),
    ('repo', 'exam', 'other'),
    ('repo', 'exam paper', 'other'),
    ('repo', 'examen', 'other'),
    ('repo', 'exegesis', 'other'),
    ('repo', 'exhibition', 'other'),
    ('repo', 'exhibition / performance', 'other'),
    ('repo', 'exhibition view', 'other'),
    ('repo', 'exhibition/performance/recreation', 'other'),
    ('repo', 'experiment', 'other'),
    ('repo', 'external research report', 'report'),
    ('repo', 'facilities', 'other'),
    ('repo', 'fact sheet', 'report'),
    ('repo', 'factsheet', 'report'),
    ('repo', 'fagmodulprojekt', 'other'),
    ('repo', 'fascículos de periódicos', 'other'),
    ('repo', 'fashion, costume and jewelry', 'other'),
    ('repo', 'feature story', 'other'),
    ('repo', 'fi=b3 vertaisarvioimaton artikkeli konferenssijulkaisussa|sv=b3 icke-referentgranskad artikel i konferenspublikation|en=b3 non-refereed conference proceedings|', 'conference-paper'),
    ('repo', 'fi=d2 artikkeli ammatillisessa kokoomateoksessa (ml. toimittajan kirjoittama johdantoartikkeli)|sv=d2 artikel i ett yrkesinriktat samlingsverk (inkl. inledningsartikel som skrivits av redaktören)|en=d2 article in a professional book (incl. an introduction by the editor)|', 'book-chapter'),
    ('repo', 'fi=d4 julkaistu kehittämis- tai tutkimusraportti taikka -selvitys|sv=d4 publicerad utvecklings- eller forskningsrapport eller -utredning|en=d4 published development or research report or study|', 'report'),
    ('repo', 'fi=d5 ammatillinen kirja|sv=d5 yrkesinriktad bok|en=d5 textbook, professional manual or guide|', 'book'),
    ('repo', 'fi=d6 toimitettu ammatillinen teos|sv=d6 redigerat yrkesinriktat verk|en= d6 edited professional book|', 'book'),
    ('repo', 'fi=kirja|sv=bok|en=book|', 'other'),
    ('repo', 'fiction', 'other'),
    ('repo', 'figurine', 'other'),
    ('repo', 'file', 'other'),
    ('repo', 'film', 'other'),
    ('repo', 'film, audio, video and digital art', 'other'),
    ('repo', 'film, audio, video and digital art; sculpture and installations', 'other'),
    ('repo', 'film/video', 'other'),
    ('repo', 'financial record', 'other'),
    ('repo', 'financial records', 'other'),
    ('repo', 'finding aid', 'other'),
    ('repo', 'findingaid', 'other'),
    ('repo', 'first cycle, g2e', 'dissertation'),
    ('repo', 'flugblatt', 'other'),
    ('repo', 'flugblätt', 'other'),
    ('repo', 'folder', 'other'),
    ('repo', 'folders', 'other'),
    ('repo', 'folheto', 'other'),
    ('repo', 'folhetos', 'other'),
    ('repo', 'folyóiratcikk - journal article', 'other'),
    ('repo', 'fondo patrimonial', 'other'),
    ('repo', 'forschungsbericht', 'report'),
    ('repo', 'forskningsrapport', 'other'),
    ('repo', 'frontmatter', 'other'),
    ('repo', 'full_issue', 'other'),
    ('repo', 'fullissue', 'other'),
    ('repo', 'furniture', 'other'),
    ('repo', 'g1 kandidaatintyö', 'dissertation'),
    ('repo', 'g2 pro gradu, diplomityö', 'dissertation'),
    ('repo', 'gallery view', 'other'),
    ('repo', 'gazeta', 'other'),
    ('repo', 'gazety', 'other'),
    ('repo', 'generic research data', 'other'),
    ('repo', 'genomic data', 'other'),
    ('repo', 'geologic atlas', 'other'),
    ('repo', 'government record', 'other'),
    ('repo', 'graduating project', 'dissertation'),
    ('repo', 'grafika', 'other'),
    ('repo', 'graphic design and illustration', 'other'),
    ('repo', 'graphic project', 'other'),
    ('repo', 'gravações de som', 'other'),
    ('repo', 'grb', 'other'),
    ('repo', 'guia docent', 'other'),
    ('repo', 'h1', 'other'),
    ('repo', 'h2', 'other'),
    ('repo', 'h3', 'other'),
    ('repo', 'habilitation', 'dissertation'),
    ('repo', 'habilitation à diriger des recherches', 'dissertation'),
    ('repo', 'habilitationsschriften', 'other'),
    ('repo', 'habilitační práce', 'dissertation'),
    ('repo', 'hak cipta', 'other'),
    ('repo', 'handcrafted item', 'other'),
    ('repo', 'handskrift', 'other'),
    ('repo', 'honors project', 'other'),
    ('repo', 'hos', 'other'),
    ('repo', 'house', 'other'),
    ('repo', 'houses', 'other'),
    ('repo', 'housing_court_decision', 'other'),
    ('repo', 'howto', 'other'),
    ('repo', 'http://purl.org/coar/resource_type/c_12cc', 'other'),
    ('repo', 'http://purl.org/coar/resource_type/c_12cd', 'other'),
    ('repo', 'http://purl.org/coar/resource_type/c_12ce', 'other'),
    ('repo', 'http://purl.org/coar/resource_type/c_15cd', 'other'),
    ('repo', 'http://purl.org/coar/resource_type/c_1843', 'other'),
    ('repo', 'http://purl.org/coar/resource_type/c_18cc', 'other'),
    ('repo', 'http://purl.org/coar/resource_type/c_18cd', 'other'),
    ('repo', 'http://purl.org/coar/resource_type/c_18cf', 'other'),
    ('repo', 'http://purl.org/coar/resource_type/c_18hj', 'other'),
    ('repo', 'http://purl.org/coar/resource_type/c_7a1f', 'other'),
    ('repo', 'http://purl.org/coar/resource_type/c_8544', 'other'),
    ('repo', 'http://purl.org/coar/resource_type/c_8a7e', 'other'),
    ('repo', 'http://purl.org/coar/resource_type/c_998f', 'other'),
    ('repo', 'http://purl.org/coar/resource_type/c_c513', 'other'),
    ('repo', 'http://purl.org/coar/resource_type/c_e059', 'other'),
    ('repo', 'http://purl.org/coar/resource_type/c_ecc8', 'other'),
    ('repo', 'http://purl.org/eprint/type/bookitem', 'book-chapter'),
    ('repo', 'http://purl.org/eprint/type/report', 'report'),
    ('repo', 'http://purl.org/redcol/resource_type/tp', 'other'),
    ('repo', 'https://vocabularies.coar-repositories.org/resource_types/c_46ec/', 'dissertation'),
    ('repo', 'humanities and social sciences', 'other'),
    ('repo', 'ihs series', 'other'),
    ('repo', 'ikonografia', 'other'),
    ('repo', 'illumination / bible - octateuch', 'other'),
    ('repo', 'illumination / profane', 'other'),
    ('repo', 'illumination / religious', 'other'),
    ('repo', 'illust', 'other'),
    ('repo', 'illustration', 'other'),
    ('repo', 'image, text', 'other'),
    ('repo', 'image-jpeg', 'other'),
    ('repo', 'image/jp2', 'other'),
    ('repo', 'image; photographs', 'other'),
    ('repo', 'image; photographs; postcards', 'other'),
    ('repo', 'image; postcards', 'other'),
    ('repo', 'image; still image', 'other'),
    ('repo', 'image; stillimage', 'other'),
    ('repo', 'image; stillimage; text;', 'other'),
    ('repo', 'image;stillimage;', 'other'),
    ('repo', 'image; stillimage', 'other'),
    ('repo', 'imagen', 'other'),
    ('repo', 'imagen en movimiento', 'other'),
    ('repo', 'imagen fija', 'other'),
    ('repo', 'images', 'other'),
    ('repo', 'imatge', 'other'),
    ('repo', 'imatge fixa', 'other'),
    ('repo', 'impreso', 'other'),
    ('repo', 'inaugural lecture', 'other'),
    ('repo', 'index', 'other'),
    ('repo', 'industry studies working paper', 'other'),
    ('repo', 'info:ar-repo/semantics/libro', 'book'),
    ('repo', 'info:eu-repo/semantics/', 'other'),
    ('repo', 'info:eu-repo/semantics/annotation', 'other'),
    ('repo', 'info:eu-repo/semantics/book chapter', 'book-chapter'),
    ('repo', 'info:eu-repo/semantics/book_part', 'book-chapter'),
    ('repo', 'info:eu-repo/semantics/bookchapter', 'book-chapter'),
    ('repo', 'info:eu-repo/semantics/bookreview', 'book-review'),
    ('repo', 'info:eu-repo/semantics/broadcast', 'other'),
    ('repo', 'info:eu-repo/semantics/conference object', 'conference-paper'),
    ('repo', 'info:eu-repo/semantics/conferencecontribution', 'conference-paper'),
    ('repo', 'info:eu-repo/semantics/conferenceposter', 'conference-abstract'),
    ('repo', 'info:eu-repo/semantics/conferenceproceedings', 'conference-paper'),
    ('repo', 'info:eu-repo/semantics/dataset', 'dataset'),
    ('repo', 'info:eu-repo/semantics/doctoral', 'dissertation'),
    ('repo', 'info:eu-repo/semantics/journal', 'other'),
    ('repo', 'info:eu-repo/semantics/learningobject', 'other'),
    ('repo', 'info:eu-repo/semantics/libro', 'book'),
    ('repo', 'info:eu-repo/semantics/movingimage', 'other'),
    ('repo', 'info:eu-repo/semantics/patent', 'other'),
    ('repo', 'info:eu-repo/semantics/periodical', 'other'),
    ('repo', 'info:eu-repo/semantics/periodicalpart', 'other'),
    ('repo', 'info:eu-repo/semantics/preprint', 'preprint'),
    ('repo', 'info:eu-repo/semantics/reporte', 'report'),
    ('repo', 'info:eu-repo/semantics/reportpart', 'report'),
    ('repo', 'info:eu-repo/semantics/researchdata', 'dataset'),
    ('repo', 'info:eu-repo/semantics/review', 'review'),
    ('repo', 'info:eu-repo/semantics/software', 'software'),
    ('repo', 'info:eu-repo/semantics/technicaldocumentation', 'report'),
    ('repo', 'info:eu-repo/semantics/video', 'other'),
    ('repo', 'info:eu-repo/semantics/working paper', 'report'),
    ('repo', 'info:eu-repo/semantics/workingpaper', 'report'),
    ('repo', 'info:pe-repo/semantics/stillimage', 'other'),
    ('repo', 'info:ulb-repo/semantics/openurl/vlink-dissertation', 'dissertation'),
    ('repo', 'informator', 'other'),
    ('repo', 'informe', 'report'),
    ('repo', 'informe científico', 'other'),
    ('repo', 'informe de gestión', 'other'),
    ('repo', 'informe técnico', 'report'),
    ('repo', 'inkunabuł', 'book'),
    ('repo', 'inne', 'other'),
    ('repo', 'installation', 'other'),
    ('repo', 'installation art', 'other'),
    ('repo', 'installation view', 'other'),
    ('repo', 'installation, environmental, stage sets', 'other'),
    ('repo', 'institucionaldocument', 'other'),
    ('repo', 'instructional material', 'other'),
    ('repo', 'interactive media element (ime)', 'other'),
    ('repo', 'interactive resource', 'other'),
    ('repo', 'internal report', 'report'),
    ('repo', 'internet contribution', 'other'),
    ('repo', 'internet publication', 'other'),
    ('repo', 'internship', 'other'),
    ('repo', 'internship report', 'report'),
    ('repo', 'interviews', 'other'),
    ('repo', 'intro', 'other'),
    ('repo', 'introduction', 'other'),
    ('repo', 'invitedpresentation', 'other'),
    ('repo', 'invoice', 'other'),
    ('repo', 'isirev', 'peer-review'),
    ('repo', 'isolated ronde-bosse / bronze - full-length statue', 'other'),
    ('repo', 'isolated ronde-bosse / bronze - full-length statue (statuette)', 'other'),
    ('repo', 'isolated ronde-bosse / plaster', 'other'),
    ('repo', 'isolated ronde-bosse / stone - full-length statue', 'other'),
    ('repo', 'isolated ronde-bosse / stone - heads and busts', 'other'),
    ('repo', 'isolated ronde-bosse / terracotta - heads and busts (vase with head of a black)', 'other'),
    ('repo', 'isolated ronde-bosse / wood - full-length statue', 'other'),
    ('repo', 'issue', 'paratext'),
    ('repo', 'journal (paginated)', 'other'),
    ('repo', 'journal issue', 'other'),
    ('repo', 'journal item', 'other'),
    ('repo', 'journal/magazine/newsletter', 'other'),
    ('repo', 'journal/periodic publication', 'other'),
    ('repo', 'journal_issue', 'other'),
    ('repo', 'journal_volume', 'other'),
    ('repo', 'journals (periodicals)', 'other'),
    ('repo', 'jpeg', 'other'),
    ('repo', 'jpeg2000', 'other'),
    ('repo', 'julkaisu', 'other'),
    ('repo', 'kalendarz', 'other'),
    ('repo', 'kalendarze', 'other'),
    ('repo', 'kandidatprojekt', 'other'),
    ('repo', 'katalog', 'other'),
    ('repo', 'kausijulkaisu', 'other'),
    ('repo', 'kirja', 'book'),
    ('repo', 'kirjat ja pienpainatteet', 'other'),
    ('repo', 'konferenz- oder workshop-beitrag', 'conference-paper'),
    ('repo', 'konferenzbeitrag', 'conference-paper'),
    ('repo', 'konferenzveröffentlichung', 'conference-paper'),
    ('repo', 'korespondencje', 'letter'),
    ('repo', 'kronika', 'other'),
    ('repo', 'książka', 'book'),
    ('repo', 'kurzbericht', 'other'),
    ('repo', 'kézirat', 'other'),
    ('repo', 'könyv', 'book'),
    ('repo', 'könyv része', 'book-chapter'),
    ('repo', 'könyvrészlet - book section', 'other'),
    ('repo', 'l2', 'other'),
    ('repo', 'l3', 'other'),
    ('repo', 'lainnya', 'other'),
    ('repo', 'laporan kkn', 'report'),
    ('repo', 'laporan kp', 'report'),
    ('repo', 'laporan mtp', 'other'),
    ('repo', 'law review', 'other'),
    ('repo', 'learning material', 'other'),
    ('repo', 'learningobject', 'other'),
    ('repo', 'lecture notes', 'other'),
    ('repo', 'lecture à haute voix', 'other'),
    ('repo', 'lectures', 'other'),
    ('repo', 'legal document', 'other'),
    ('repo', 'legislative document', 'other'),
    ('repo', 'legislação', 'other'),
    ('repo', 'lehtiartikkeli', 'other'),
    ('repo', 'lesson_plan', 'other'),
    ('repo', 'let', 'other'),
    ('repo', 'letter to the editor', 'letter'),
    ('repo', 'letters (correspondence)', 'letter'),
    ('repo', 'libro', 'book'),
    ('repo', 'linguistic type: language description', 'other'),
    ('repo', 'linguistic type: lexicon', 'reference-entry'),
    ('repo', 'linguistic type: primary text', 'other'),
    ('repo', 'list', 'other'),
    ('repo', 'list gratulacyjny', 'other'),
    ('repo', 'lists', 'other'),
    ('repo', 'listy', 'other'),
    ('repo', 'lithographs', 'other'),
    ('repo', 'livre', 'book'),
    ('repo', 'livro', 'book'),
    ('repo', 'livros', 'book'),
    ('repo', 'llibre', 'book'),
    ('repo', 'locdec', 'other'),
    ('repo', 'lítico', 'other'),
    ('repo', 'm1', 'other'),
    ('repo', 'm2', 'other'),
    ('repo', 'm3', 'other'),
    ('repo', 'magazin', 'other'),
    ('repo', 'magazine', 'other'),
    ('repo', 'magazine and newsletter', 'other'),
    ('repo', 'magazine article', 'other'),
    ('repo', 'main resource of the dataset', 'other'),
    ('repo', 'makale', 'other'),
    ('repo', 'manual', 'other'),
    ('repo', 'manuscript map', 'other'),
    ('repo', 'manuscripts', 'other'),
    ('repo', 'manuscritos', 'other'),
    ('repo', 'map', 'other'),
    ('repo', 'map or cartographic material', 'other'),
    ('repo', 'mapa', 'other'),
    ('repo', 'mapas', 'other'),
    ('repo', 'maps', 'other'),
    ('repo', 'maps; prints', 'other'),
    ('repo', 'mask', 'other'),
    ('repo', 'master', 'dissertation'),
    ('repo', 'master in biochemistry and molecular and cellular biology', 'other'),
    ('repo', 'master in biology', 'dissertation'),
    ('repo', 'master in business engineering professional focus in analytics &amp; digital business', 'other'),
    ('repo', 'master in business engineering professional focus in data science', 'other'),
    ('repo', 'master in computer science', 'dissertation'),
    ('repo', 'master in economy: general, professional focus', 'other'),
    ('repo', 'master in management', 'dissertation'),
    ('repo', 'master in management professional focus', 'other'),
    ('repo', 'master in management professional focus in business analysis &amp; integration', 'other'),
    ('repo', 'master in mathematics', 'other'),
    ('repo', 'master in pharmaceutical sciences, professionnal focus', 'other'),
    ('repo', 'master of philosophy', 'dissertation'),
    ('repo', 'master of philosophy (mphil)', 'other'),
    ('repo', 'master of science', 'other'),
    ('repo', 'master of science by research (mscr)', 'other'),
    ('repo', 'master projects', 'other'),
    ('repo', 'master\'s paper', 'other'),
    ('repo', 'master\'s project', 'other'),
    ('repo', 'master\'s report', 'dissertation'),
    ('repo', 'masterarbeit', 'dissertation'),
    ('repo', 'masterprogrammepaper', 'other'),
    ('repo', 'masters', 'dissertation'),
    ('repo', 'masters project', 'other'),
    ('repo', 'masters term project', 'other'),
    ('repo', 'masters_capstone_project', 'other'),
    ('repo', 'mastersproject', 'other'),
    ('repo', 'maszynopis', 'other'),
    ('repo', 'material didáctico', 'other'),
    ('repo', 'material docente', 'other'),
    ('repo', 'material impreso', 'other'),
    ('repo', 'materiales sonoros', 'other'),
    ('repo', 'materials', 'other'),
    ('repo', 'matériel de conférence', 'conference-paper'),
    ('repo', 'media', 'other'),
    ('repo', 'meet', 'other'),
    ('repo', 'meeting abstract', 'conference-abstract'),
    ('repo', 'meetings', 'other'),
    ('repo', 'meetings and proceedings', 'conference-paper'),
    ('repo', 'memorandum', 'report'),
    ('repo', 'meny', 'other'),
    ('repo', 'metalwork', 'other'),
    ('repo', 'methodical recommendations', 'other'),
    ('repo', 'midas-objekt', 'other'),
    ('repo', 'miesięcznik', 'other'),
    ('repo', 'military atlas', 'other'),
    ('repo', 'mini dissertation', 'dissertation'),
    ('repo', 'mini review', 'review'),
    ('repo', 'miniature', 'other'),
    ('repo', 'minspe', 'other'),
    ('repo', 'minutes', 'other'),
    ('repo', 'miscellaneous documents', 'other'),
    ('repo', 'mixed materials', 'other'),
    ('repo', 'mixed media', 'other'),
    ('repo', 'mon', 'other'),
    ('repo', 'mot', 'other'),
    ('repo', 'motion', 'other'),
    ('repo', 'moving image;', 'other'),
    ('repo', 'movingimage', 'other'),
    ('repo', 'mpra paper', 'report'),
    ('repo', 'mrp', 'other'),
    ('repo', 'multilingual poetry', 'other'),
    ('repo', 'multimedia', 'other'),
    ('repo', 'multiple', 'other'),
    ('repo', 'multivolume_work', 'book'),
    ('repo', 'mural painting / antiquity', 'other'),
    ('repo', 'mural painting / viii-xx century', 'other'),
    ('repo', 'museum', 'other'),
    ('repo', 'museumsobjekt', 'other'),
    ('repo', 'music item', 'other'),
    ('repo', 'musical composition', 'other'),
    ('repo', 'musical compositions', 'other'),
    ('repo', 'musical performances', 'other'),
    ('repo', 'musical score', 'other'),
    ('repo', 'musical score/notation', 'other'),
    ('repo', 'musicprogram', 'other'),
    ('repo', 'mémoire', 'dissertation'),
    ('repo', 'mémoire accepté', 'dissertation'),
    ('repo', 'mémoire de maîtrise', 'dissertation'),
    ('repo', 'mémoire ou thèse', 'dissertation'),
    ('repo', 'na', 'other'),
    ('repo', 'national atlas', 'other'),
    ('repo', 'negative (photographic)', 'other'),
    ('repo', 'negatives', 'other'),
    ('repo', 'nekrolog', 'other'),
    ('repo', 'new books', 'other'),
    ('repo', 'news article', 'other'),
    ('repo', 'news item/press item', 'other'),
    ('repo', 'newsletter', 'other'),
    ('repo', 'newspaper / magazine', 'other'),
    ('repo', 'newspaper or magazine article', 'other'),
    ('repo', 'newspaper/magazine article', 'other'),
    ('repo', 'newspaper;', 'other'),
    ('repo', 'newspapers', 'other'),
    ('repo', 'niitype:others', 'other'),
    ('repo', 'niitype:technical report', 'report'),
    ('repo', 'nitrate film', 'other'),
    ('repo', 'non traditional textual works', 'other'),
    ('repo', 'non-refereed case study', 'other'),
    ('repo', 'non_textual', 'other'),
    ('repo', 'nonfiction', 'other'),
    ('repo', 'nonpeerreviewed', 'other'),
    ('repo', 'notated music', 'other'),
    ('repo', 'notcrit', 'other'),
    ('repo', 'note', 'other'),
    ('repo', 'notes on geographic distribution', 'other'),
    ('repo', 'notícia de jornal', 'other'),
    ('repo', 'num', 'other'),
    ('repo', 'numeric', 'other'),
    ('repo', 'nuty', 'other'),
    ('repo', 'números', 'other'),
    ('repo', 'objecte d\'aprenentatge', 'other'),
    ('repo', 'objeto de aprendizaje', 'other'),
    ('repo', 'objeto de conferencia', 'conference-paper'),
    ('repo', 'objeto fisico', 'other'),
    ('repo', 'objeto_virtual_de_aprendizaje_ova', 'other'),
    ('repo', 'obraz', 'other'),
    ('repo', 'office documents', 'other'),
    ('repo', 'online publication', 'other'),
    ('repo', 'online resource', 'other'),
    ('repo', 'opinion', 'editorial'),
    ('repo', 'opinion pieces / media / blogs', 'other'),
    ('repo', 'opis bibliograficzny', 'other'),
    ('repo', 'opracowanie', 'other'),
    ('repo', 'oral histories', 'other'),
    ('repo', 'oral history', 'other'),
    ('repo', 'oral paper presentation', 'other'),
    ('repo', 'oral presentation', 'other'),
    ('repo', 'oral_presentation', 'other'),
    ('repo', 'original creative work (portfolio)', 'other'),
    ('repo', 'original creative works - textual work', 'other'),
    ('repo', 'other conference contributions', 'other'),
    ('repo', 'other form of assessable output', 'other'),
    ('repo', 'other literature type', 'other'),
    ('repo', 'other periodical', 'other'),
    ('repo', 'other_doctype', 'other'),
    ('repo', 'other_document', 'other'),
    ('repo', 'others', 'other'),
    ('repo', 'otras publicaciones periódicas', 'other'),
    ('repo', 'otro', 'other'),
    ('repo', 'otros', 'other'),
    ('repo', 'outputmanagementplan', 'other'),
    ('repo', 'ouvrage', 'book'),
    ('repo', 'ouvrec', 'other'),
    ('repo', 'painting', 'other'),
    ('repo', 'painting on sundry supports', 'other'),
    ('repo', 'paintings', 'other'),
    ('repo', 'paintings; drawings and watercolors', 'other'),
    ('repo', 'paintings; sculpture and installations', 'other'),
    ('repo', 'paintings; works on paper', 'other'),
    ('repo', 'pamiętnik', 'other'),
    ('repo', 'pamphlet', 'other'),
    ('repo', 'pamphlets', 'other'),
    ('repo', 'panel', 'other'),
    ('repo', 'panorama', 'other'),
    ('repo', 'papers in conference proceedings', 'other'),
    ('repo', 'parole_document', 'other'),
    ('repo', 'parte de libro', 'other'),
    ('repo', 'parte de livro', 'other'),
    ('repo', 'pasantía', 'other'),
    ('repo', 'past year examination question', 'other'),
    ('repo', 'pastel', 'other'),
    ('repo', 'patente', 'other'),
    ('repo', 'patents', 'other'),
    ('repo', 'patient information leaflet', 'other'),
    ('repo', 'patri', 'other'),
    ('repo', 'pdf', 'other'),
    ('repo', 'performance', 'other'),
    ('repo', 'performance art', 'other'),
    ('repo', 'performing arts (including performance art)', 'other'),
    ('repo', 'periodical', 'other'),
    ('repo', 'periodical part', 'other'),
    ('repo', 'periodicalpart', 'other'),
    ('repo', 'periodicals', 'other'),
    ('repo', 'periódico', 'other'),
    ('repo', 'periódicos y revistas', 'other'),
    ('repo', 'personal correspondence', 'letter'),
    ('repo', 'perspective', 'other'),
    ('repo', 'pesquisa bibliográfica', 'review'),
    ('repo', 'phd dissertation', 'dissertation'),
    ('repo', 'phd doctor of philosophy', 'dissertation'),
    ('repo', 'phd/doctoral dissertation', 'dissertation'),
    ('repo', 'photograph;', 'other'),
    ('repo', 'photographs; architecture and city planning', 'other'),
    ('repo', 'photographs; black and white photograph', 'other'),
    ('repo', 'photographs; black-and-white photographs', 'other'),
    ('repo', 'photographs; decorative arts, utilitarian objects and interior design', 'other'),
    ('repo', 'photographs; performing arts (including performance art)', 'other'),
    ('repo', 'photographs; sculpture and installations', 'other'),
    ('repo', 'photographs|negative', 'other'),
    ('repo', 'photography', 'other'),
    ('repo', 'photomechanical prints', 'other'),
    ('repo', 'photos', 'other'),
    ('repo', 'physicalojbect', 'other'),
    ('repo', 'pidato guru besar', 'other'),
    ('repo', 'pintura', 'other'),
    ('repo', 'plakat', 'conference-abstract'),
    ('repo', 'plan', 'other'),
    ('repo', 'plan de estudios', 'other'),
    ('repo', 'plan de la réussite', 'other'),
    ('repo', 'plan or blueprint', 'other'),
    ('repo', 'plan stratégique', 'other'),
    ('repo', 'pocztówka', 'other'),
    ('repo', 'podcast', 'other'),
    ('repo', 'podium_presentation', 'other'),
    ('repo', 'podiumpresentation', 'other'),
    ('repo', 'podręczniki', 'other'),
    ('repo', 'poem', 'other'),
    ('repo', 'poetry', 'other'),
    ('repo', 'policy document', 'report'),
    ('repo', 'policy note', 'other'),
    ('repo', 'policy paper', 'report'),
    ('repo', 'policy report', 'report'),
    ('repo', 'politique', 'other'),
    ('repo', 'ponencia', 'conference-paper'),
    ('repo', 'popular press / news item', 'other'),
    ('repo', 'portfolio', 'other'),
    ('repo', 'portrait photographs', 'other'),
    ('repo', 'post-print', 'preprint'),
    ('repo', 'postcard', 'other'),
    ('repo', 'postcards', 'other'),
    ('repo', 'poster (research)', 'other'),
    ('repo', 'poster of condolence', 'other'),
    ('repo', 'poster presentation', 'conference-abstract'),
    ('repo', 'posters', 'conference-abstract'),
    ('repo', 'postkarte', 'other'),
    ('repo', 'praca doktorska', 'dissertation'),
    ('repo', 'praca dyplomowa', 'other'),
    ('repo', 'practice', 'other'),
    ('repo', 'practice tool', 'other'),
    ('repo', 'pre-print', 'preprint'),
    ('repo', 'preface', 'other'),
    ('repo', 'presentación', 'other'),
    ('repo', 'presentation / conference', 'conference-paper'),
    ('repo', 'presentation / conference contribution', 'conference-paper'),
    ('repo', 'press release', 'other'),
    ('repo', 'press releases', 'other'),
    ('repo', 'prezentace', 'other'),
    ('repo', 'prezentacja', 'other'),
    ('repo', 'print', 'other'),
    ('repo', 'print / engraving on copper - illustration', 'other'),
    ('repo', 'print / engraving on copper - isolated print', 'other'),
    ('repo', 'print / engraving on wood - illustration', 'other'),
    ('repo', 'print / engraving on wood - isolated print', 'other'),
    ('repo', 'print / lithograph - illustration', 'other'),
    ('repo', 'print / lithograph - isolated print', 'other'),
    ('repo', 'printed publication', 'other'),
    ('repo', 'prints', 'other'),
    ('repo', 'prints; cartoons', 'other'),
    ('repo', 'pro gradu-tutkielma', 'dissertation'),
    ('repo', 'problem statement, exercise', 'other'),
    ('repo', 'proceeding', 'conference-paper'),
    ('repo', 'proceeding paper', 'other'),
    ('repo', 'proceedings paper', 'conference-paper'),
    ('repo', 'proceedingspaper', 'other'),
    ('repo', 'professional masters project', 'other'),
    ('repo', 'professionalpaper', 'other'),
    ('repo', 'professionalpapercoa', 'other'),
    ('repo', 'program', 'other'),
    ('repo', 'program teatralny', 'other'),
    ('repo', 'programas de mano de teatro  -murcia', 'other'),
    ('repo', 'programme', 'other'),
    ('repo', 'programoverviews', 'other'),
    ('repo', 'programy nauczania', 'other'),
    ('repo', 'project deliverable', 'report'),
    ('repo', 'project paper report', 'other'),
    ('repo', 'project report', 'report'),
    ('repo', 'project reports', 'other'),
    ('repo', 'projecte/treball final de carrera', 'dissertation'),
    ('repo', 'prose fiction', 'other'),
    ('repo', 'protein-ligand binding data', 'dataset'),
    ('repo', 'proyecto', 'other'),
    ('repo', 'proyecto aplicado', 'other'),
    ('repo', 'proyecto aplicado o tesis', 'dissertation'),
    ('repo', 'proyecto de investigacion', 'other'),
    ('repo', 'proyecto de investigación', 'other'),
    ('repo', 'proyecto fin de carrera', 'dissertation'),
    ('repo', 'proyecto_de_investigacion', 'other'),
    ('repo', 'präsentation auf konferenz', 'conference-abstract'),
    ('repo', 'publicacion seriada', 'other'),
    ('repo', 'publicació en sèrie', 'other'),
    ('repo', 'publication', 'other'),
    ('repo', 'publication - book section', 'book-chapter'),
    ('repo', 'publication - conference item', 'conference-paper'),
    ('repo', 'publication - report', 'report'),
    ('repo', 'publication de collège', 'other'),
    ('repo', 'publication gouvernementale ou paragouvernementale', 'report'),
    ('repo', 'publicação didática', 'other'),
    ('repo', 'publicity photograph', 'other'),
    ('repo', 'publikacja pokonferencyjna', 'conference-paper'),
    ('repo', 'published article or volume', 'other'),
    ('repo', 'published research report', 'report'),
    ('repo', 'pàgines web', 'other'),
    ('repo', 'pòster de congrés', 'other'),
    ('repo', 'póster', 'conference-abstract'),
    ('repo', 'póster de congreso', 'conference-abstract'),
    ('repo', 'quality control scan', 'other'),
    ('repo', 'radio program', 'other'),
    ('repo', 'radio programs', 'other'),
    ('repo', 'raport', 'report'),
    ('repo', 'rapport', 'report'),
    ('repo', 'rapport de recherche', 'report'),
    ('repo', 'rapporter', 'report'),
    ('repo', 'real estate map', 'other'),
    ('repo', 'realia', 'other'),
    ('repo', 'recenzja', 'other'),
    ('repo', 'recenzja rozprawy doktorskiej', 'other'),
    ('repo', 'record (document)', 'other'),
    ('repo', 'recording', 'other'),
    ('repo', 'recording, acoustical', 'other'),
    ('repo', 'recording, musical', 'other'),
    ('repo', 'recording, oral', 'other'),
    ('repo', 'refer', 'other'),
    ('repo', 'refereed', 'other'),
    ('repo', 'reference', 'reference-entry'),
    ('repo', 'reference material', 'reference-entry'),
    ('repo', 'reference work', 'other'),
    ('repo', 'regional atlas', 'other'),
    ('repo', 'registration', 'other'),
    ('repo', 'relatório de estágio', 'report'),
    ('repo', 'relatório de projeto', 'other'),
    ('repo', 'relatório técnico', 'report'),
    ('repo', 'religious buildings', 'other'),
    ('repo', 'rep', 'other'),
    ('repo', 'report (commissioned)', 'other'),
    ('repo', 'report or working paper', 'report'),
    ('repo', 'report, research', 'other'),
    ('repo', 'report, technical', 'other'),
    ('repo', 'report:report', 'other'),
    ('repo', 'reporte', 'report'),
    ('repo', 'reports and papers', 'other'),
    ('repo', 'reports, briefing/ working papers', 'other'),
    ('repo', 'research paper (m.a.), 3 hrs.', 'dissertation'),
    ('repo', 'research paper (m.a.), 4 hrs.', 'dissertation'),
    ('repo', 'research paper (m.a.e.), 4 hrs.', 'other'),
    ('repo', 'research paper (m.s.), 3 hrs.', 'other'),
    ('repo', 'research paper or report', 'report'),
    ('repo', 'research report (external)', 'other'),
    ('repo', 'research reports', 'report'),
    ('repo', 'research reports or papers', 'other'),
    ('repo', 'research_article', 'other'),
    ('repo', 'research_data', 'dataset'),
    ('repo', 'researchdata', 'dataset'),
    ('repo', 'resenha', 'book-review'),
    ('repo', 'reseña', 'other'),
    ('repo', 'reseña de libro', 'book-review'),
    ('repo', 'reseña libro', 'book-review'),
    ('repo', 'residential buildings', 'other'),
    ('repo', 'resource', 'other'),
    ('repo', 'response', 'other'),
    ('repo', 'ressenya', 'book-review'),
    ('repo', 'resumen', 'conference-abstract'),
    ('repo', 'resumo de comunicação em conferência internacional', 'other'),
    ('repo', 'resumo de comunicação em conferência nacional', 'other'),
    ('repo', 'resumo publicado em evento', 'conference-abstract'),
    ('repo', 'rev', 'other'),
    ('repo', 'reviewarticle', 'review'),
    ('repo', 'revista', 'other'),
    ('repo', 'revista divulgativa', 'other'),
    ('repo', 'revisão de literatura', 'review'),
    ('repo', 'rezension', 'book-review'),
    ('repo', 'road atlas', 'other'),
    ('repo', 'rocznik', 'other'),
    ('repo', 'ronde-bosse / gold and silverwork - miscellaneous objects', 'other'),
    ('repo', 'rozdział', 'other'),
    ('repo', 'rozdział w książce', 'book-chapter'),
    ('repo', 'rozdział z książki', 'book-chapter'),
    ('repo', 'rozprawa doktorska', 'dissertation'),
    ('repo', 'rysunek', 'other'),
    ('repo', 'rękopis', 'other'),
    ('repo', 'rękopis muzyczny', 'other'),
    ('repo', 'rękopisy', 'other'),
    ('repo', 'sa_bill', 'other'),
    ('repo', 'sachakte', 'other'),
    ('repo', 'sammelwerk', 'book'),
    ('repo', 'sammelwerksbeitrag', 'book-chapter'),
    ('repo', 'sc_agenda', 'other'),
    ('repo', 'sc_minutes', 'other'),
    ('repo', 'school atlas', 'other'),
    ('repo', 'script', 'other'),
    ('repo', 'sculpture', 'other'),
    ('repo', 'sculpture and installations', 'other'),
    ('repo', 'sculpture and installations; architecture and city planning', 'other'),
    ('repo', 'sculpture and installations; drawings and watercolors', 'other'),
    ('repo', 'sculpture and installations; film, audio, video and digital art', 'other'),
    ('repo', 'sculpture and installations; paintings', 'other'),
    ('repo', 'sculpture and installations; photographs', 'other'),
    ('repo', 'sculpture and installations; prints', 'other'),
    ('repo', 'sección de libro', 'book-chapter'),
    ('repo', 'second cycle, a2e', 'dissertation'),
    ('repo', 'semesterprojekt', 'other'),
    ('repo', 'seminar paper', 'other'),
    ('repo', 'seminario', 'other'),
    ('repo', 'senior project', 'dissertation'),
    ('repo', 'separate map', 'other'),
    ('repo', 'serial', 'other'),
    ('repo', 'series', 'other'),
    ('repo', 'sheet', 'other'),
    ('repo', 'short communication', 'other'),
    ('repo', 'short report', 'report'),
    ('repo', 'short stories', 'other'),
    ('repo', 'shortstory', 'other'),
    ('repo', 'show / exhibition', 'other'),
    ('repo', 'show, exhibition or event', 'other'),
    ('repo', 'show/exhibition', 'other'),
    ('repo', 'sidewall', 'other'),
    ('repo', 'siegel', 'other'),
    ('repo', 'single-family dwelling', 'other'),
    ('repo', 'sketches', 'other'),
    ('repo', 'skripsi', 'dissertation'),
    ('repo', 'skrypt', 'other'),
    ('repo', 'slide', 'other'),
    ('repo', 'software or program code', 'other'),
    ('repo', 'solicitud de patente', 'other'),
    ('repo', 'son inédit', 'other'),
    ('repo', 'song & music', 'other'),
    ('repo', 'sonstige', 'other'),
    ('repo', 'sonstige veröffentlichung', 'other'),
    ('repo', 'sonstiges', 'other'),
    ('repo', 'sotatieteiden kandidaattiopiskelijan tutkielma', 'other'),
    ('repo', 'sotatieteiden maisteriopiskelijan pro gradu', 'other'),
    ('repo', 'sound;', 'other'),
    ('repo', 'sound; text', 'other'),
    ('repo', 'sound; text;', 'other'),
    ('repo', 'specialist', 'other'),
    ('repo', 'specimen', 'other'),
    ('repo', 'spee', 'other'),
    ('repo', 'speech', 'other'),
    ('repo', 'speeches/lectures', 'other'),
    ('repo', 'spis', 'other'),
    ('repo', 'sprawozdania', 'report'),
    ('repo', 'sprawozdanie', 'report'),
    ('repo', 'sprawozdanie szkolne xix-xx w.', 'report'),
    ('repo', 'stained glass', 'other'),
    ('repo', 'standard or specification', 'other'),
    ('repo', 'starodruk', 'book'),
    ('repo', 'starodruki', 'other'),
    ('repo', 'stary druk', 'book'),
    ('repo', 'state_exam', 'other'),
    ('repo', 'statement', 'other'),
    ('repo', 'statistical atlas', 'other'),
    ('repo', 'statistiques', 'other'),
    ('repo', 'statt', 'other'),
    ('repo', 'statut', 'other'),
    ('repo', 'statystyki', 'other'),
    ('repo', 'stellungnahme', 'editorial'),
    ('repo', 'still image;', 'other'),
    ('repo', 'still image; text', 'other'),
    ('repo', 'still images', 'other'),
    ('repo', 'still images.photograph', 'other'),
    ('repo', 'still_image', 'other'),
    ('repo', 'stpetersburg', 'other'),
    ('repo', 'streszczenie rozprawy doktorskiej', 'other'),
    ('repo', 'strona dydaktyczna', 'other'),
    ('repo', 'strona naukowa', 'other'),
    ('repo', 'strona naukowo-dydaktyczna', 'other'),
    ('repo', 'strona popularnonaukowa', 'other'),
    ('repo', 'student project', 'other'),
    ('repo', 'student research project', 'other'),
    ('repo', 'student works', 'other'),
    ('repo', 'student_research', 'dissertation'),
    ('repo', 'studenthandbooks', 'other'),
    ('repo', 'studijní materiál', 'other'),
    ('repo', 'study', 'other'),
    ('repo', 'su ūkio subjektais', 'other'),
    ('repo', 'subject guide', 'other'),
    ('repo', 'subscription_content', 'other'),
    ('repo', 'summary', 'other'),
    ('repo', 'survey', 'other'),
    ('repo', 'syllabus', 'other'),
    ('repo', 'symposium', 'conference-paper'),
    ('repo', 'synopsis', 'other'),
    ('repo', 'szkice', 'other'),
    ('repo', 'tabular data', 'other'),
    ('repo', 'tagungsband', 'other'),
    ('repo', 'taikomieji moksliniai tyrimai / applied research (tmt)', 'other'),
    ('repo', 'talk', 'other'),
    ('repo', 'tarjeta postal', 'other'),
    ('repo', 'taxonomy & inventories', 'other'),
    ('repo', 'tcc', 'dissertation'),
    ('repo', 'tcc (graduação)', 'dissertation'),
    ('repo', 'tcces', 'other'),
    ('repo', 'tccgrad', 'dissertation'),
    ('repo', 'teaching', 'other'),
    ('repo', 'teaching resource', 'other'),
    ('repo', 'teaching_aid', 'other'),
    ('repo', 'technical drawing', 'other'),
    ('repo', 'technical note', 'report'),
    ('repo', 'technical paper', 'other'),
    ('repo', 'technical reports', 'report'),
    ('repo', 'technical_report', 'report'),
    ('repo', 'technicalreport', 'other'),
    ('repo', 'techreport', 'report'),
    ('repo', 'tekst', 'other'),
    ('repo', 'teksti', 'other'),
    ('repo', 'term paper', 'other'),
    ('repo', 'terminal project', 'other'),
    ('repo', 'tese', 'dissertation'),
    ('repo', 'tese (doutorado)', 'dissertation'),
    ('repo', 'tesi di laurea', 'other'),
    ('repo', 'text, image', 'other'),
    ('repo', 'text, other', 'other'),
    ('repo', 'text.serial.journal', 'other'),
    ('repo', 'text::bericht', 'other'),
    ('repo', 'text::buch', 'other'),
    ('repo', 'text:in_proceedings', 'conference-paper'),
    ('repo', 'text;', 'other'),
    ('repo', 'text; dataset', 'other'),
    ('repo', 'text; image', 'other'),
    ('repo', 'text; image; dataset', 'other'),
    ('repo', 'text; image; stillimage', 'other'),
    ('repo', 'text; periodicals', 'other'),
    ('repo', 'text; sound', 'other'),
    ('repo', 'text; still image', 'other'),
    ('repo', 'textile', 'other'),
    ('repo', 'textile / tapestry', 'other'),
    ('repo', 'textiles', 'other'),
    ('repo', 'texto', 'other'),
    ('repo', 'texto de apresentação/encerramento', 'other'),
    ('repo', 'texto impreso', 'other'),
    ('repo', 'textual work', 'other'),
    ('repo', 'tez', 'dissertation'),
    ('repo', 'thematic atlas', 'other'),
    ('repo', 'theses / dissertations', 'dissertation'),
    ('repo', 'theses and dissertations', 'dissertation'),
    ('repo', 'thèse d\'exercice', 'dissertation'),
    ('repo', 'thèse de doctorat', 'dissertation'),
    ('repo', 'thèse et mémoire', 'dissertation'),
    ('repo', 'thèse ou essai doctoral accepté', 'dissertation'),
    ('repo', 'thèse ou mémoire de l\'uqac', 'dissertation'),
    ('repo', 'thèse ou mémoire de l\'uqar', 'other'),
    ('repo', 'thèse ou mémoires', 'other'),
    ('repo', 'thèse, mémoire ou essai', 'dissertation'),
    ('repo', 'thèses de doctorat', 'dissertation'),
    ('repo', 'tool', 'other'),
    ('repo', 'trabajo de grado', 'dissertation'),
    ('repo', 'trabajo de grado - doctorado', 'dissertation'),
    ('repo', 'trabajo de grado - especialización', 'dissertation'),
    ('repo', 'trabajo de grado - maestría', 'dissertation'),
    ('repo', 'trabajo de grado - pregrado', 'dissertation'),
    ('repo', 'trabajo de grado, licenciatura', 'dissertation'),
    ('repo', 'trabajo de grado, maestría', 'other'),
    ('repo', 'trabajo de suficiencia profesional', 'dissertation'),
    ('repo', 'trabajo fin de grado/gradu amaierako lana', 'dissertation'),
    ('repo', 'trabajo final de grado', 'dissertation'),
    ('repo', 'trabajo recepcional', 'other'),
    ('repo', 'trabajo revisado (peer-reviewed)', 'other'),
    ('repo', 'trabajo terminal, especialidad', 'other'),
    ('repo', 'trabajos de grado', 'other'),
    ('repo', 'trabalho apresentado em evento', 'conference-paper'),
    ('repo', 'trabalho completo publicado em evento', 'conference-paper'),
    ('repo', 'trabalho de conclusão de especialização', 'dissertation'),
    ('repo', 'trabalho de conclusão de graduação', 'dissertation'),
    ('repo', 'trabalho apresentado em evento', 'conference-paper'),
    ('repo', 'training material', 'other'),
    ('repo', 'trainingworkshop', 'other'),
    ('repo', 'transcript', 'other'),
    ('repo', 'transcription', 'other'),
    ('repo', 'translation', 'other'),
    ('repo', 'travail dirigé (document diplômant)', 'other'),
    ('repo', 'treball acadèmic', 'other'),
    ('repo', 'treball de fi de postgrau', 'dissertation'),
    ('repo', 'treball de recerca', 'dissertation'),
    ('repo', 'treball final de grau', 'dissertation'),
    ('repo', 'tugas akhir', 'other'),
    ('repo', 'tutkimusjulkaisu', 'other'),
    ('repo', 'tutorial', 'other'),
    ('repo', 'tygodniki', 'other'),
    ('repo', 'tygodniki ilustrowane', 'other'),
    ('repo', 'type:editorial', 'other'),
    ('repo', 'type:history', 'other'),
    ('repo', 'type:informatics', 'other'),
    ('repo', 'type:math', 'other'),
    ('repo', 'type:news', 'other'),
    ('repo', 'type:other', 'other'),
    ('repo', 'type:physics', 'other'),
    ('repo', 'type:review', 'review'),
    ('repo', 'ulotka', 'other'),
    ('repo', 'undergraduate theses', 'other'),
    ('repo', 'university record', 'other'),
    ('repo', 'unspecified', 'other'),
    ('repo', 'upm news', 'other'),
    ('repo', 'urban landscape', 'other'),
    ('repo', 'valokuva', 'other'),
    ('repo', 'veranstaltungsbeitrag (unveröffentlicht)', 'conference-paper'),
    ('repo', 'video', 'other'),
    ('repo', 'video projection still', 'other'),
    ('repo', 'video recording', 'other'),
    ('repo', 'video still', 'other'),
    ('repo', 'video/moving image', 'other'),
    ('repo', 'video: documentales', 'other'),
    ('repo', 'video: entrevistas', 'other'),
    ('repo', 'videograbacion', 'other'),
    ('repo', 'videograbación', 'other'),
    ('repo', 'virtual-oral-presentation', 'other'),
    ('repo', 'visual media', 'other'),
    ('repo', 'visual works: paintings: mural paintings', 'other'),
    ('repo', 'visual works: sculpture: reliefs: bas-reliefs; visual works: sculpture: rock carvings: rock engravings', 'other'),
    ('repo', 'vlink-patent', 'other'),
    ('repo', 'vlink-unknown', 'other'),
    ('repo', 'volume (multivolumework)', 'other'),
    ('repo', 'volume (periodical)', 'other'),
    ('repo', 'volume/issue', 'other'),
    ('repo', 'volume_manuscript', 'other'),
    ('repo', 'vídeo', 'other'),
    ('repo', 'wall map', 'other'),
    ('repo', 'war posters', 'other'),
    ('repo', 'watercolor', 'other'),
    ('repo', 'web page capture', 'other'),
    ('repo', 'web product', 'other'),
    ('repo', 'website', 'other'),
    ('repo', 'website content', 'other'),
    ('repo', 'wiersz', 'other'),
    ('repo', 'window installation', 'other'),
    ('repo', 'wood engraving', 'other'),
    ('repo', 'working / discussion paper', 'report'),
    ('repo', 'working document', 'report'),
    ('repo', 'working or discussion paper', 'report'),
    ('repo', 'working paper / technical report', 'other'),
    ('repo', 'working paper/technical report', 'report'),
    ('repo', 'working/technical paper', 'report'),
    ('repo', 'working_paper', 'report'),
    ('repo', 'works on paper', 'other'),
    ('repo', 'workshop', 'conference-paper'),
    ('repo', 'world atlas', 'other'),
    ('repo', 'wspomnienie', 'other'),
    ('repo', 'wydawnictwa urzędowe', 'other'),
    ('repo', 'yearbooks', 'other'),
    ('repo', 'yleisesikuntaupseerikurssin opiskelijan diplomityö', 'other'),
    ('repo', 'yüksek lisans', 'dissertation'),
    ('repo', 'zai', 'other'),
    ('repo', 'zaproszenie', 'other'),
    ('repo', 'zeitschrif', 'other'),
    ('repo', 'zeitschriften', 'other'),
    ('repo', 'zeitung', 'other'),
    ('repo', 'zestaw danych', 'other'),
    ('repo', 'áudio', 'other'),
    ('repo', 'авторефераты диссертаций', 'other'),
    ('repo', 'дипломски рад', 'other'),
    ('repo', 'дисертація', 'other'),
    ('repo', 'диссертации', 'other'),
    ('repo', 'доповідь на конференції або симпозіумі', 'other'),
    ('repo', 'доповідь на конференції чи семінарі', 'conference-paper'),
    ('repo', 'експеримент', 'other'),
    ('repo', 'изоиздания', 'other'),
    ('repo', 'кваліфікаційні роботи здобувачів', 'other'),
    ('repo', 'книга', 'book'),
    ('repo', 'книга, монография (book)', 'book'),
    ('repo', 'магистарска теза', 'other'),
    ('repo', 'мастер рад', 'other'),
    ('repo', 'монографии', 'other'),
    ('repo', 'монографія', 'other'),
    ('repo', 'навчальний матеріал', 'other'),
    ('repo', 'навчально-методичні матеріали', 'other'),
    ('repo', 'научный доклад (working paper)', 'report'),
    ('repo', 'поглавље у монографији', 'other'),
    ('repo', 'підручник/посібник', 'book'),
    ('repo', 'рад у зборнику', 'conference-paper'),
    ('repo', 'саопштење са скупа штампано у изводу', 'other'),
    ('repo', 'сборники', 'other'),
    ('repo', 'спеціалізовані вчені ради', 'other'),
    ('repo', 'статьи в сборниках', 'conference-paper'),
    ('repo', 'тезисы доклада (abstracts)', 'other'),
    ('repo', 'учебные издания', 'other'),
    ('repo', 'учебные материалы', 'other'),
    ('repo', 'інший', 'other'),
    ('repo', 'ללא', 'other'),
    ('repo', 'מפות', 'other'),
    ('repo', 'תרשימי תכנון', 'other'),
    ('repo', 'წიგნი', 'book'),
    ('repo', '专著', 'book'),
    ('repo', '专著章节, 文集论文', 'other'),
    ('repo', '会议论文', 'conference-paper'),
    ('repo', '其他', 'other'),
    ('repo', '博士', 'dissertation'),
    ('repo', '失效', 'other'),
    ('repo', '学位论文', 'dissertation'),
    ('repo', '应用软件', 'other'),
    ('repo', '授权', 'other'),
    ('repo', '文章', 'other'),
    ('repo', '新闻', 'other'),
    ('repo', '有效', 'other'),
    ('repo', '硕士', 'dissertation'),
    ('repo', '编著', 'book'),
    ('repo', '青年科学基金项目', 'other'),
    ('repo', '面上项目', 'other'),
    ('repo', '项目', 'other'),
    ('repo', '기술용역보고서', 'report'),
    ('repo', '서울시 간행물', 'other'),
    ('repo', '학술용역보고서', 'report'),
    ('datacite', 'audiovisual', 'other'),
    ('datacite', 'award', 'other'),
    ('datacite', 'book', 'book'),
    ('datacite', 'bookchapter', 'book-chapter'),
    ('datacite', 'collection', 'other'),
    ('datacite', 'computationalnotebook', 'software'),
    ('datacite', 'conferencepaper', 'conference-paper'),
    ('datacite', 'conferenceproceeding', 'conference-paper'),
    ('datacite', 'datapaper', 'data-paper'),
    ('datacite', 'dataset', 'dataset'),
    ('datacite', 'dissertation', 'dissertation'),
    ('datacite', 'event', 'other'),
    ('datacite', 'image', 'other'),
    ('datacite', 'instrument', 'other'),
    ('datacite', 'interactiveresource', 'other'),
    ('datacite', 'journal', 'other'),
    ('datacite', 'journalarticle', 'article'),
    ('datacite', 'model', 'dataset'),
    ('datacite', 'modeloutput', 'other'),
    ('datacite', 'other', 'other'),
    ('datacite', 'peerreview', 'peer-review'),
    ('datacite', 'physicalobject', 'other'),
    ('datacite', 'poster', 'conference-abstract'),
    ('datacite', 'preprint', 'preprint'),
    ('datacite', 'projectreport', 'report'),
    ('datacite', 'report', 'report'),
    ('datacite', 'service', 'other'),
    ('datacite', 'software', 'software'),
    ('datacite', 'sound', 'other'),
    ('datacite', 'standard', 'standard'),
    ('datacite', 'studyregistration', 'other'),
    ('datacite', 'text', 'article'),
    ('datacite', 'workflow', 'other'),
    ('datacite', 'chapter', 'book-chapter'),
    ('datacite', 'thesis', 'dissertation'),
    ('crossref', 'book', 'book'),
    ('crossref', 'book-chapter', 'book-chapter'),
    ('crossref', 'book-part', 'book-chapter'),
    ('crossref', 'book-series', 'paratext'),
    ('crossref', 'book-set', 'book'),
    ('crossref', 'book-track', 'book-chapter'),
    ('crossref', 'dataset', 'dataset'),
    ('crossref', 'dissertation', 'dissertation'),
    ('crossref', 'edited-book', 'book'),
    ('crossref', 'journal', 'paratext'),
    ('crossref', 'journal-issue', 'paratext'),
    ('crossref', 'journal-volume', 'paratext'),
    ('crossref', 'monograph', 'book'),
    ('crossref', 'other', 'other'),
    ('crossref', 'peer-review', 'peer-review'),
    ('crossref', 'proceedings', 'paratext'),
    ('crossref', 'proceedings-series', 'paratext'),
    ('crossref', 'reference-book', 'book'),
    ('crossref', 'reference-entry', 'reference-entry'),
    ('crossref', 'report', 'report'),
    ('crossref', 'report-series', 'paratext'),
    ('crossref', 'standard', 'standard'),
    ('pubmed', 'address', 'other'),
    ('pubmed', 'autobiography', 'other'),
    ('pubmed', 'bibliography', 'paratext'),
    ('pubmed', 'biography', 'other'),
    ('pubmed', 'classical article', 'other'),
    ('pubmed', 'clinical conference', 'other'),
    ('pubmed', 'collected work', 'other'),
    ('pubmed', 'comment', 'letter'),
    ('pubmed', 'congress', 'paratext'),
    ('pubmed', 'consensus development conference', 'other'),
    ('pubmed', 'corrected and republished article', 'erratum'),
    ('pubmed', 'dataset', 'dataset'),
    ('pubmed', 'dictionary', 'paratext'),
    ('pubmed', 'directory', 'paratext'),
    ('pubmed', 'duplicate publication', 'other'),
    ('pubmed', 'editorial', 'editorial'),
    ('pubmed', 'electronic supplementary materials', 'supplementary-materials'),
    ('pubmed', 'english abstract', 'other'),
    ('pubmed', 'expression of concern', 'other'),
    ('pubmed', 'festschrift', 'other'),
    ('pubmed', 'government publication', 'other'),
    ('pubmed', 'guideline', 'other'),
    ('pubmed', 'historical article', 'other'),
    ('pubmed', 'interactive tutorial', 'other'),
    ('pubmed', 'interview', 'other'),
    ('pubmed', 'introductory journal article', 'other'),
    ('pubmed', 'lecture', 'other'),
    ('pubmed', 'legal case', 'other'),
    ('pubmed', 'legislation', 'other'),
    ('pubmed', 'letter', 'letter'),
    ('pubmed', 'meta-analysis', 'review'),
    ('pubmed', 'news', 'other'),
    ('pubmed', 'newspaper article', 'other'),
    ('pubmed', 'overall', 'other'),
    ('pubmed', 'patient education handout', 'other'),
    ('pubmed', 'peer review', 'peer-review'),
    ('pubmed', 'periodical index', 'paratext'),
    ('pubmed', 'personal narrative', 'other'),
    ('pubmed', 'portrait', 'other'),
    ('pubmed', 'practice guideline', 'other'),
    ('pubmed', 'preprint', 'preprint'),
    ('pubmed', 'published erratum', 'erratum'),
    ('pubmed', 'research support, american recovery and reinvestment act', 'other'),
    ('pubmed', 'research support, n.i.h., extramural', 'other'),
    ('pubmed', 'research support, n.i.h., intramural', 'other'),
    ('pubmed', 'research support, non-u.s. gov\'t', 'other'),
    ('pubmed', 'research support, u.s. gov\'t, non-p.h.s.', 'other'),
    ('pubmed', 'research support, u.s. gov\'t, p.h.s.', 'other'),
    ('pubmed', 'retracted publication', 'retraction'),
    ('pubmed', 'retraction of publication', 'retraction'),
    ('pubmed', 'review', 'review'),
    ('pubmed', 'scientific integrity review', 'review'),
    ('pubmed', 'systematic review', 'review'),
    ('pubmed', 'technical report', 'report'),
    ('pubmed', 'video-audio media', 'other'),
    ('pubmed', 'webcast', 'other')) AS t(family, k, mapped_type)
)
SELECT l.* EXCEPT (type),
  CASE WHEN pw.preprint_registrant THEN 'preprint' WHEN sc.cascade_rule = 'default: -> article' THEN coalesce(dm.mapped_type, nullif(l.type, ''), 'article') ELSE sc.cascade_type END AS type,  -- THE FLIP: classifier verdict IS the type
  CASE WHEN pw.preprint_registrant THEN 'preprint' WHEN sc.cascade_rule = 'default: -> article' THEN coalesce(dm.mapped_type, nullif(l.type, ''), 'article') ELSE sc.cascade_type END AS classified_type,
  CASE WHEN pw.preprint_registrant THEN 'preprint-registrant DOI prefix' WHEN sc.cascade_rule = 'default: -> article' AND dm.mapped_type IS NOT NULL THEN concat('ingest-dict fallback: ', dm.family) WHEN sc.cascade_rule = 'default: -> article' AND nullif(l.type, '') IS NOT NULL THEN 'ingest-type preserved' ELSE sc.cascade_rule END AS classified_rule
FROM identifier('openalex' || :env_suffix || '.works.locations_w_sources') l
JOIN scored sc ON sc.work_id = concat_ws('~', l.provenance, l.native_id_namespace, l.native_id)
JOIN (SELECT work_id AS pw_id, preprint_registrant FROM works) pw
  ON pw.pw_id = sc.work_id
LEFT JOIN dict_map dm
  ON dm.family = CASE WHEN l.provenance IN ('repo', 'repo_backfill') THEN 'repo'
                      WHEN l.provenance = 'datacite' THEN 'datacite'
                      WHEN l.provenance = 'crossref' THEN 'crossref'
                      WHEN l.provenance = 'pubmed' THEN 'pubmed' END
  AND dm.k = lower(coalesce(l.raw_type, ''))
);

In [0]:
SELECT classified_type, count(*) AS n
FROM identifier('openalex' || :env_suffix || '.works.locations_w_types')
GROUP BY 1 ORDER BY n DESC LIMIT 30;